# Flatsubcon đa seed trên Vast.ai

Notebook này ghi source cần thiết vào `PROJECT_DIR`, tải đúng snapshot dữ liệu PAP_NER, rồi chạy SupCon và linear probe độc lập cho từng seed. Chọn backbone ở cell cấu hình đầu tiên.

**Tái lập:** cùng seed vẫn có thể lệch nhỏ do FP16 và CUDA atomics; vì vậy báo cáo kết quả nhiều seed theo mean ± std.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile

# =========================
# BACKBONE REGISTRY
# =========================
MODEL_REGISTRY = {
    "xlm_r": {"model_name_or_path": "FacebookAI/xlm-roberta-base", "use_fast_tokenizer": False},
    "phobert_base": {"model_name_or_path": "vinai/phobert-base", "use_fast_tokenizer": False},
    "electra": {"model_name_or_path": "NlpHUST/electra-base-vn", "use_fast_tokenizer": False},
    "bamibert": {"model_name_or_path": "Qualcomm-AI-Research/BamiBERT", "use_fast_tokenizer": True},
    "distilphobert": {"model_name_or_path": "trungbb8/distilphobert", "use_fast_tokenizer": False},
}
MODEL_CHOICE = "xlm_r"
if MODEL_CHOICE not in MODEL_REGISTRY:
    raise ValueError(f"MODEL_CHOICE không hợp lệ: {MODEL_CHOICE}; chọn một trong {list(MODEL_REGISTRY)}")
SELECTED_MODEL = MODEL_REGISTRY[MODEL_CHOICE]

# =========================
# CẤU HÌNH THỰC NGHIỆM — giữ đúng baseline_linear_eval.ipynb
# =========================
SEEDS = [42, 43, 44, 45, 46]
STRICT_DETERMINISM = False
EFFECTIVE_BATCH_SIZE = 64
MAX_MICRO_BATCH_SIZE = 64  # Tự giảm khi CUDA OOM; effective batch vẫn là 64.
MAX_SEQ_LENGTH = 256
NUM_WORKERS = 2
STAGE1_CONFIG = {
    "batch_size": 64,
    "learning_rate": 2e-5,
    "head_learning_rate": 5e-4,
    "supcon_temp": 0.07,
    "projection_dim": 128,
    "epochs": 15,
    "early_stop": 5,
    "fp16": True,
    "warmup_proportion": 0.1,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "max_seq_length": 256,
    "num_workers": 2,
}
STAGE2_CONFIG = {
    "batch_size": 64,
    "classifier_learning_rate": 1e-3,
    "epochs": 20,
    "early_stop": 5,
    "fp16": True,
    "freeze_backbone": True,
    "max_seq_length": 256,
    "num_workers": 2,
}
TRAINING_CONFIG = {"stage1": STAGE1_CONFIG, "stage2": STAGE2_CONFIG}
assert SEEDS == [42, 43, 44, 45, 46]
assert EFFECTIVE_BATCH_SIZE == 64 and MAX_MICRO_BATCH_SIZE <= EFFECTIVE_BATCH_SIZE
assert STAGE1_CONFIG == {
    "batch_size": 64, "learning_rate": 2e-5, "head_learning_rate": 5e-4,
    "supcon_temp": 0.07, "projection_dim": 128, "epochs": 15, "early_stop": 5,
    "fp16": True, "warmup_proportion": 0.1, "weight_decay": 0.01,
    "max_grad_norm": 1.0, "max_seq_length": 256, "num_workers": 2,
}
assert STAGE2_CONFIG == {
    "batch_size": 64, "classifier_learning_rate": 1e-3, "epochs": 20,
    "early_stop": 5, "fp16": True, "freeze_backbone": True,
    "max_seq_length": 256, "num_workers": 2,
}
assert STAGE1_CONFIG["batch_size"] == STAGE2_CONFIG["batch_size"] == EFFECTIVE_BATCH_SIZE
assert EFFECTIVE_BATCH_SIZE % MAX_MICRO_BATCH_SIZE == 0

# =========================
# ĐƯỜNG DẪN TUYỆT ĐỐI
# =========================
WORK_ROOT = Path(os.environ.get("FLATSUBCON_WORK_ROOT", "/workspace/flatsubcon_multiseed_vast")).expanduser().resolve()
PROJECT_DIR = (WORK_ROOT / "project").resolve()
RUN_ROOT = (WORK_ROOT / "runs").resolve()
HF_DATASET_ROOT = (WORK_ROOT / "hf-datasets").resolve()
HF_HOME = (WORK_ROOT / "hf-cache").resolve()
ZIP_PATH = (WORK_ROOT / "flatsubcon_multiseed_results.zip").resolve()
DATASET_DIR = (PROJECT_DIR / "Dataset").resolve()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
HF_DATASET_ROOT.mkdir(parents=True, exist_ok=True)

HF_DATASET_REPO = "levandai1704/PAP_NER_dedup_70_15_15"
HF_DATASET_REVISION = "e4af303d22f878e5827c94861f50f747a6b3ed45"
EXPECTED_SENTENCES = {"train": 24089, "dev": 5162, "test": 5162}
INCLUDE_CHECKPOINTS = False
ENV = dict(os.environ)
ENV.update({
    "HF_HOME": str(HF_HOME),
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONUNBUFFERED": "1",
    "PYTHONIOENCODING": "utf-8",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "MPLCONFIGDIR": str((WORK_ROOT / "matplotlib-cache").resolve()),
})
Path(ENV["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

print(f"Model: {MODEL_CHOICE} ({SELECTED_MODEL['model_name_or_path']})")
print(f"Seeds: {SEEDS}; effective batch: {EFFECTIVE_BATCH_SIZE}; strict determinism: {STRICT_DETERMINISM}")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"RUN_ROOT: {RUN_ROOT}")

## 1. Cài đặt và kiểm tra Python/GPU
Cài pandas trước khi import; các package được cài qua Python subprocess, không dùng notebook magics.

In [ ]:
# pandas được cài trước khi import trong kernel.
# Kernel Vast có thể chưa có PyTorch: cài đúng bản dùng ở các notebook softmax (2.5.1 + CUDA 12.1).
import importlib.util
if importlib.util.find_spec("torch") is None:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "torch==2.5.1",
        "--index-url", "https://download.pytorch.org/whl/cu121",
    ], check=True)
    importlib.invalidate_caches()

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pandas==2.2.3"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "numpy<2.0", "scikit-learn==1.5.2", "matplotlib>=3.8,<4", "tqdm",
    "transformers==4.48.3", "huggingface_hub==0.29.3", "tokenizers==0.21.0",
    "sentencepiece", "protobuf<6",
], check=True)

nerve = subprocess.run(
    [sys.executable, "-m", "pip", "install", "nervaluate==1.2.1"],
    check=False, text=True,
)
if nerve.returncode:
    print("Không cài được nervaluate 1.2.1; evaluator Strict exact span/type dự phòng sẽ dùng.")

import pandas as pd
import numpy as np
import torch
import transformers
from importlib.metadata import PackageNotFoundError, version

print("Python:", sys.version.replace("\n", " "))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("pandas:", pd.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
try:
    print("nervaluate:", version("nervaluate"))
except PackageNotFoundError:
    print("nervaluate: not installed; exact span/type fallback is active if needed")

## 2. Ghi toàn bộ source vào PROJECT_DIR
Các file không đổi được ghi nguyên văn từ repo mẫu; các file khác đã được điều chỉnh cho tokenizer/backbone tổng quát, seed ổn định và phần đánh giá test theo yêu cầu.

In [ ]:
SOURCE_FILES = {
    'main.py': 'import os\nimport sys\n\n# Đảm bảo đường dẫn thư mục hiện tại được ưu tiên hàng đầu trong sys.path\n_script_dir = os.path.dirname(os.path.abspath(__file__))\nif _script_dir not in sys.path:\n    sys.path.insert(0, _script_dir)\n\nfrom src.constant import LOGGER\nfrom src.arguments import get_stage1_argument, get_stage2_argument, get_eval_argument\nfrom src.trainer import train_stage1_supcon, train_stage2_linear, evaluate_linear_probe\nfrom src.models import PhoBertLinearProbe\nfrom src.dataset import build_dataset\nfrom src.helper import safe_torch_load\nfrom transformers import AutoTokenizer, AutoConfig\nfrom torch.utils.data import DataLoader, SequentialSampler\nimport torch\n\n\ndef main():\n    if len(sys.argv) < 2:\n        print("=" * 70)\n        print("📌 GIAO THỨC LINEAR EVALUATION CHO PAP_NER (PhoBERT + SupCon)")\n        print("Cách sử dụng:")\n        print("  1. Huấn luyện Giai đoạn 1 (Pure Contrastive):")\n        print("     python main.py stage1_supcon --data_dir Dataset --epochs 15")\n        print("  2. Huấn luyện Giai đoạn 2 (Linear Probing Evaluation, Freeze Backbone, NO CRF):")\n        print("     python main.py stage2_linear --data_dir Dataset --epochs 20")\n        print("  3. Đánh giá Checkpoint Giai đoạn 2 trên tập Test:")\n        print("     python main.py evaluate --model_path outputs/stage2_linear/best_linear_probe.pt")\n        print("  4. Chạy toàn bộ cả 2 Giai đoạn tự động:")\n        print("     python main.py full_pipeline")\n        print("=" * 70)\n        sys.exit(0)\n\n    mode = sys.argv[1].lower()\n    # Loại bỏ tên chế độ khỏi sys.argv để ArgumentParser parse các cờ còn lại\n    sys.argv.pop(1)\n\n    if mode == \'stage1_supcon\':\n        args = get_stage1_argument()\n        train_stage1_supcon(args)\n\n    elif mode == \'stage2_linear\':\n        args = get_stage2_argument()\n        train_stage2_linear(args)\n\n    elif mode == \'full_pipeline\':\n        LOGGER.info("🚀 BẮT ĐẦU CHẠY TOÀN BỘ PIPELINE (STAGE 1 -> STAGE 2)...")\n        # 1. Chạy Stage 1\n        args_stage1 = get_stage1_argument()\n        best_encoder_path = train_stage1_supcon(args_stage1)\n\n        # 2. Chạy Stage 2 sử dụng encoder vừa huấn luyện\n        args_stage2 = get_stage2_argument()\n        args_stage2.pretrained_encoder_path = best_encoder_path\n        train_stage2_linear(args_stage2)\n\n    elif mode == \'evaluate\':\n        args = get_eval_argument()\n        device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        LOGGER.info(f"Nạp checkpoint đánh giá: {args.model_path}")\n        checkpoint = safe_torch_load(args.model_path, map_location=device)\n        config = AutoConfig.from_pretrained(args.model_name_or_path)\n        tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path, use_fast=False)\n\n        model = PhoBertLinearProbe(config, num_labels=11, freeze_backbone=True).to(device)\n        model.load_state_dict(checkpoint[\'model_state_dict\'])\n\n        from src.constant import LABEL2ID_PAP_NER, LABEL_MAPPING\n        test_dataset = build_dataset(\n            data_dir=args.data_dir,\n            tokenizer=tokenizer,\n            label2id=LABEL2ID_PAP_NER,\n            header=LABEL_MAPPING[\'pap_ner\'][\'header\'],\n            dtype=\'test\',\n            max_seq_len=args.max_seq_length\n        )\n        test_loader = DataLoader(\n            test_dataset,\n            sampler=SequentialSampler(test_dataset),\n            batch_size=args.batch_size,\n            num_workers=args.num_workers\n        )\n        evaluate_linear_probe(model, test_loader, device=device, is_test=True, output_dir=args.output_dir)\n\n    else:\n        LOGGER.error(f"❌ Chế độ \'{mode}\' không hợp lệ! Vui lòng chọn: stage1_supcon, stage2_linear, full_pipeline, evaluate.")\n\n\nif __name__ == \'__main__\':\n    main()\n\n',
    'src/__init__.py': "from .constant import LOGGER, LABEL2ID_PAP_NER, LABEL_MAPPING\nfrom .helper import set_random_seed, safe_torch_load, init_logger, seed_worker, make_generator\nfrom .arguments import get_stage1_argument, get_stage2_argument, get_eval_argument\nfrom .dataset import build_dataset, NerDataset\nfrom .conlleval import evaluate\nfrom .models import (\n    ProjectionHead, TokenSupConLoss, TokenEncoderSupCon, TokenEncoderLinearProbe,\n    SupConOutput, LinearProbeOutput\n)\nfrom .trainer import train_stage1_supcon, train_stage2_linear, evaluate_linear_probe\n\n__all__ = [\n    'LOGGER', 'LABEL2ID_PAP_NER', 'LABEL_MAPPING', 'set_random_seed', 'safe_torch_load',\n    'init_logger', 'seed_worker', 'make_generator', 'get_stage1_argument',\n    'get_stage2_argument', 'get_eval_argument', 'build_dataset', 'NerDataset', 'evaluate',\n    'ProjectionHead', 'TokenSupConLoss', 'TokenEncoderSupCon', 'TokenEncoderLinearProbe',\n    'SupConOutput', 'LinearProbeOutput', 'train_stage1_supcon', 'train_stage2_linear',\n    'evaluate_linear_probe'\n]\n",
    'src/arguments.py': 'import argparse\n\n\ndef get_stage1_argument():\n    parser = argparse.ArgumentParser(description="Stage 1: Pure Supervised Contrastive Pre-training (Google SupContrast 100%, NO ALPHA)")\n    parser.add_argument("--data_dir", default="Dataset", type=str,\n                        help="The input data dir. Should contain train_data.txt, dev_data.txt, test_data.txt")\n    parser.add_argument("--output_dir", default="outputs/stage1_supcon", type=str,\n                        help="Where to save Stage 1 pre-trained encoder checkpoint")\n    parser.add_argument("--model_name_or_path", default="trungbb8/distilphobert", type=str,\n                        help="Pre-trained PhoBERT model: trungbb8/distilphobert, vinai/phobert-base, vinai/phobert-large")\n    parser.add_argument("--max_seq_length", default=256, type=int,\n                        help="Maximum sequence length")\n    parser.add_argument("--batch_size", default=64, type=int,\n                        help="Batch size for SupCon training")\n    parser.add_argument("--eval_batch_size", default=64, type=int,\n                        help="Batch size for validation")\n    parser.add_argument("--learning_rate", default=2e-5, type=float,\n                        help="Learning rate for PhoBERT backbone in Stage 1")\n    parser.add_argument("--head_learning_rate", default=5e-4, type=float,\n                        help="Learning rate for Projection Head in Stage 1")\n    parser.add_argument("--supcon_temp", default=0.07, type=float,\n                        help="Temperature tau for SupCon loss (default: 0.07)")\n    parser.add_argument("--projection_dim", default=128, type=int,\n                        help="Dimension of contrastive hypersphere representation")\n    parser.add_argument("--epochs", default=15, type=int,\n                        help="Number of epochs for Stage 1")\n    parser.add_argument("--warmup_proportion", default=0.1, type=float,\n                        help="Warmup proportion for linear schedule")\n    parser.add_argument("--weight_decay", default=0.01, type=float,\n                        help="Weight decay")\n    parser.add_argument("--max_grad_norm", default=1.0, type=float,\n                        help="Max gradient norm")\n    parser.add_argument("--early_stop", default=5, type=int,\n                        help="Early stopping patience based on Validation SupCon loss")\n    parser.add_argument("--fp16", action="store_true", default=True,\n                        help="Use mixed precision training (AMP)")\n    parser.add_argument("--no_fp16", dest="fp16", action="store_false")\n    parser.add_argument("--seed", default=42, type=int,\n                        help="Random seed")\n    parser.add_argument("--num_workers", default=2, type=int,\n                        help="DataLoader worker subprocesses")\n    parser.add_argument("--overwrite_data", action="store_true", default=False,\n                        help="Overwrite cached preprocessed datasets")\n    return parser.parse_args()\n\n\ndef get_stage2_argument():\n    parser = argparse.ArgumentParser(description="Stage 2: Linear Probing Evaluation Protocol (Freeze Backbone, Train Linear Classifier, NO CRF)")\n    parser.add_argument("--data_dir", default="Dataset", type=str,\n                        help="The input data dir.")\n    parser.add_argument("--output_dir", default="outputs/stage2_linear", type=str,\n                        help="Where to save Stage 2 Linear Probe model checkpoint")\n    parser.add_argument("--model_name_or_path", default="trungbb8/distilphobert", type=str,\n                        help="Pre-trained PhoBERT model")\n    parser.add_argument("--pretrained_encoder_path", default="outputs/stage1_supcon/best_encoder.pt", type=str,\n                        help="Path to pre-trained encoder from Stage 1 (SupCon)")\n    parser.add_argument("--max_seq_length", default=256, type=int,\n                        help="Maximum sequence length")\n    parser.add_argument("--batch_size", default=64, type=int,\n                        help="Batch size for Linear Probe training")\n    parser.add_argument("--eval_batch_size", default=64, type=int,\n                        help="Batch size for eval")\n    parser.add_argument("--classifier_learning_rate", default=1e-3, type=float,\n                        help="Learning rate for Linear Classifier (default: 1e-3)")\n    parser.add_argument("--epochs", default=20, type=int,\n                        help="Number of epochs for Stage 2")\n    parser.add_argument("--early_stop", default=5, type=int,\n                        help="Early stopping patience based on Validation Strict F1")\n    parser.add_argument("--freeze_backbone", action="store_true", default=True,\n                        help="Strict Linear Evaluation Protocol: Freeze backbone parameters (requires_grad=False)")\n    parser.add_argument("--unfreeze_backbone", dest="freeze_backbone", action="store_false",\n                        help="Optional full fine-tuning mode")\n    parser.add_argument("--fp16", action="store_true", default=True,\n                        help="Use mixed precision (AMP)")\n    parser.add_argument("--no_fp16", dest="fp16", action="store_false")\n    parser.add_argument("--seed", default=42, type=int,\n                        help="Random seed")\n    parser.add_argument("--num_workers", default=2, type=int,\n                        help="DataLoader worker subprocesses")\n    parser.add_argument("--overwrite_data", action="store_true", default=False)\n    return parser.parse_args()\n\n\ndef get_eval_argument():\n    parser = argparse.ArgumentParser(description="Evaluate Stage 2 Linear Probe on Test Set")\n    parser.add_argument("--data_dir", default="Dataset", type=str)\n    parser.add_argument("--model_path", default="outputs/stage2_linear/best_linear_probe.pt", type=str,\n                        help="Path to trained Linear Probe checkpoint")\n    parser.add_argument("--model_name_or_path", default="trungbb8/distilphobert", type=str)\n    parser.add_argument("--batch_size", default=64, type=int)\n    parser.add_argument("--max_seq_length", default=256, type=int)\n    parser.add_argument("--num_workers", default=2, type=int)\n    parser.add_argument("--output_dir", default="outputs/stage2_linear", type=str)\n    return parser.parse_args()\n',
    'src/conlleval.py': '"""\nCoNLL-2003 Official Evaluation Script for Chunking/NER Evaluation\nEvaluates Strict Entity-level Precision, Recall, and Micro-F1.\nAdapted from sighsmile/conlleval for Vietnamese PAP_NER.\n"""\n\nfrom __future__ import division, print_function, unicode_literals\n\nimport sys\nfrom collections import defaultdict\nfrom .constant import LOGGER\n\n\ndef split_tag(chunk_tag):\n    """\n    Split chunk tag into BIO prefix and chunk_type\n    e.g.\n    B-VBPL -> (\'B\', \'VBPL\')\n    I-VBPL -> (\'I\', \'VBPL\')\n    O      -> (\'O\', None)\n    """\n    if chunk_tag == \'O\':\n        return (\'O\', None)\n    parts = chunk_tag.split(\'-\', maxsplit=1)\n    if len(parts) == 2:\n        return parts[0], parts[1]\n    return (\'O\', None)\n\n\ndef is_chunk_end(prev_tag, tag):\n    prefix1, chunk_type1 = split_tag(prev_tag)\n    prefix2, chunk_type2 = split_tag(tag)\n\n    if prefix1 == \'O\':\n        return False\n    if prefix2 == \'O\':\n        return prefix1 != \'O\'\n\n    if chunk_type1 != chunk_type2:\n        return True\n\n    return prefix2 in [\'B\', \'S\'] or prefix1 in [\'E\', \'S\']\n\n\ndef is_chunk_start(prev_tag, tag):\n    prefix1, chunk_type1 = split_tag(prev_tag)\n    prefix2, chunk_type2 = split_tag(tag)\n\n    if prefix2 == \'O\':\n        return False\n    if prefix1 == \'O\':\n        return prefix2 != \'O\'\n\n    if chunk_type1 != chunk_type2:\n        return True\n\n    return prefix2 in [\'B\', \'S\'] or prefix1 in [\'E\', \'S\']\n\n\ndef calc_metrics(tp, p, t, percent=False):\n    precision = tp / p if p else 0\n    recall = tp / t if t else 0\n    fb1 = 2 * precision * recall / (precision + recall) if precision + recall else 0\n    if percent:\n        return 100 * precision, 100 * recall, 100 * fb1\n    else:\n        return precision, recall, fb1\n\n\ndef count_chunks(true_seqs, pred_seqs):\n    correct_chunks = defaultdict(int)\n    true_chunks = defaultdict(int)\n    pred_chunks = defaultdict(int)\n\n    correct_counts = defaultdict(int)\n    true_counts = defaultdict(int)\n    pred_counts = defaultdict(int)\n\n    prev_true_tag, prev_pred_tag = \'O\', \'O\'\n    in_correct = False\n\n    for true_tag, pred_tag in zip(true_seqs, pred_seqs):\n        # Check chunk ends\n        if in_correct:\n            if is_chunk_end(prev_true_tag, true_tag) and is_chunk_end(prev_pred_tag, pred_tag):\n                in_correct = False\n                correct_chunks[split_tag(prev_true_tag)[1]] += 1\n            elif is_chunk_end(prev_true_tag, true_tag) != is_chunk_end(prev_pred_tag, pred_tag):\n                in_correct = False\n\n        if is_chunk_end(prev_true_tag, true_tag):\n            true_chunks[split_tag(prev_true_tag)[1]] += 1\n        if is_chunk_end(prev_pred_tag, pred_tag):\n            pred_chunks[split_tag(prev_pred_tag)[1]] += 1\n\n        # Check chunk starts\n        if is_chunk_start(prev_true_tag, true_tag) and is_chunk_start(prev_pred_tag, pred_tag):\n            if split_tag(true_tag)[1] == split_tag(pred_tag)[1]:\n                in_correct = True\n            else:\n                in_correct = False\n\n        if is_chunk_start(prev_true_tag, true_tag):\n            true_chunks[split_tag(true_tag)[1]] += 1\n        if is_chunk_start(prev_pred_tag, pred_tag):\n            pred_chunks[split_tag(pred_tag)[1]] += 1\n\n        if true_tag == pred_tag:\n            correct_counts[true_tag] += 1\n        true_counts[true_tag] += 1\n        pred_counts[pred_tag] += 1\n\n        prev_true_tag, prev_pred_tag = true_tag, pred_tag\n\n    if in_correct:\n        correct_chunks[split_tag(prev_true_tag)[1]] += 1\n    if is_chunk_end(prev_true_tag, \'O\'):\n        true_chunks[split_tag(prev_true_tag)[1]] += 1\n    if is_chunk_end(prev_pred_tag, \'O\'):\n        pred_chunks[split_tag(prev_pred_tag)[1]] += 1\n\n    return correct_chunks, true_chunks, pred_chunks, correct_counts, true_counts, pred_counts\n\n\ndef get_result(correct_chunks, true_chunks, pred_chunks, correct_counts, true_counts, pred_counts, verbose=True):\n    sum_correct_chunks = sum(correct_chunks.values())\n    sum_true_chunks = sum(true_chunks.values())\n    sum_pred_chunks = sum(pred_chunks.values())\n\n    sum_correct_counts = sum(correct_counts.values())\n    sum_true_counts = sum(true_counts.values())\n\n    nonO_correct_counts = sum(v for k, v in correct_counts.items() if k != \'O\')\n    nonO_true_counts = sum(v for k, v in true_counts.items() if k != \'O\')\n\n    chunk_types = sorted(list(set(list(true_chunks) + list(pred_chunks))))\n\n    prec, rec, f1 = calc_metrics(sum_correct_chunks, sum_pred_chunks, sum_true_chunks, percent=False)\n    res = (prec, rec, f1)\n    if not verbose:\n        return res\n\n    LOGGER.info(f"Processed {sum_true_counts} tokens with {sum_true_chunks} gold phrases; "\n                f"Found: {sum_pred_chunks} phrases; Correct: {sum_correct_chunks}.")\n    if nonO_true_counts > 0:\n        LOGGER.info("Accuracy (excl. \'O\'): %0.4f" % (nonO_correct_counts / nonO_true_counts))\n    if sum_true_counts > 0:\n        LOGGER.info("Token Accuracy: %0.4f | Strict Precision: %0.4f | Strict Recall: %0.4f | Strict F1: %0.4f"\n                    % (sum_correct_counts / sum_true_counts, prec, rec, f1))\n    \n    for t in chunk_types:\n        p_t, r_t, f1_t = calc_metrics(correct_chunks[t], pred_chunks[t], true_chunks[t], percent=False)\n        LOGGER.info("%17s: Precision: %0.4f; Recall: %0.4f; F1-score: %0.4f (Gold: %d, Pred: %d)" \n                    % (t, p_t, r_t, f1_t, true_chunks[t], pred_chunks[t]))\n    return res\n\n\ndef evaluate(true_seqs, pred_seqs, verbose=True):\n    correct_chunks, true_chunks, pred_chunks, correct_counts, true_counts, pred_counts = count_chunks(true_seqs, pred_seqs)\n    return get_result(correct_chunks, true_chunks, pred_chunks, correct_counts, true_counts, pred_counts, verbose=verbose)\n\n',
    'src/constant.py': "import os\nfrom datetime import datetime\nfrom .helper import init_logger\n\nLOGGER = init_logger(datetime.now().strftime('linear_eval_%d%b%Y_%H-%M-%S.log'))\n\n# 11 BIO labels for Vietnamese Public Administration Procedures Named Entity Recognition (PAP_NER)\nLABEL2ID_PAP_NER = [\n    'O',\n    'B-CQ', 'I-CQ',\n    'B-ĐT', 'I-ĐT',\n    'B-VBPL', 'I-VBPL',\n    'B-NG', 'I-NG',\n    'B-SL', 'I-SL'\n]\n\nID2LABEL_PAP_NER = {idx: label for idx, label in enumerate(LABEL2ID_PAP_NER)}\n\nLABEL_MAPPING = {\n    'pap_ner': {\n        'label2id': LABEL2ID_PAP_NER,\n        'id2label': ID2LABEL_PAP_NER,\n        'header': ['token', 'pos', 'chunk', 'ner', 'tmp']\n    }\n}\n\n",
    'src/dataset.py': 'import os\nfrom pathlib import Path\nfrom typing import List, Union\nimport torch\nfrom torch.utils.data import Dataset\n\nfrom .processor import NerFeatures, convert_word_segment_examples_features\nfrom .helper import safe_torch_load\n\nif hasattr(torch.serialization, \'add_safe_globals\'):\n    try:\n        torch.serialization.add_safe_globals([NerFeatures])\n    except Exception:\n        pass\n\n\nclass NerDataset(Dataset):\n    def __init__(self, features: List[NerFeatures]):\n        self.examples = features\n\n    def __len__(self):\n        return len(self.examples)\n\n    def __getitem__(self, index):\n        return {key: val for key, val in self.examples[index].__dict__.items() if key != \'tokens\'}\n\n\ndef build_dataset(data_dir: Union[str, os.PathLike],\n                  tokenizer,\n                  label2id: List[str],\n                  header: List[str],\n                  dtype: str = \'train\',\n                  max_seq_len: int = 256,\n                  overwrite_data: bool = False) -> NerDataset:\n    """\n    Nạp và xử lý tập dữ liệu PAP_NER với cơ chế tìm kiếm đa đường dẫn\n    đảm bảo chạy ổn định tuyệt đối trên cả máy cục bộ lẫn Google Colab.\n    """\n    data_path = Path(data_dir)\n\n    # Danh sách các thư mục ứng viên để tự động tìm kiếm (Local, Colab, Google Drive)\n    candidate_dirs = [\n        data_path,\n        Path("..") / data_path,\n        Path(".") / "Dataset",\n        Path("..") / "Dataset",\n        Path("./Dataset"),\n        Path("../Dataset")\n    ]\n\n    dfile_path = None\n    searched_paths = []\n\n    for c_dir in candidate_dirs:\n        for cand in [c_dir / f"{dtype}_data.txt", c_dir / f"{dtype}.txt"]:\n            searched_paths.append(str(cand))\n            if cand.exists():\n                dfile_path = cand\n                break\n        if dfile_path is not None:\n            break\n\n    if dfile_path is None:\n        raise FileNotFoundError(\n            f"❌ Không tìm thấy tệp dữ liệu \'{dtype}\' trong các đường dẫn sau:\\n" +\n            "\\n".join(f"  - {p}" for p in searched_paths[:6]) +\n            f"\\nVui lòng kiểm tra lại tham số --data_dir hoặc đảm bảo thư mục Dataset tồn tại!"\n        )\n\n    model_tag = str(tokenizer.name_or_path).replace(\'/\', \'_\').replace(\'\\\\\', \'_\')\n    tokenizer_kind = \'fast\' if getattr(tokenizer, \'is_fast\', False) else \'slow\'\n    cached_path = dfile_path.with_name(f"{dfile_path.stem}_{model_tag}_{tokenizer_kind}_{max_seq_len}_wordalign_v3.cached")\n    legacy_cached_path = dfile_path.with_suffix(\'.cached\')\n\n    if not overwrite_data and cached_path.exists():\n        features = safe_torch_load(cached_path)\n    elif not overwrite_data and legacy_cached_path.exists() and (\'phobert-base\' in model_tag):\n        features = safe_torch_load(legacy_cached_path)\n    else:\n        features = convert_word_segment_examples_features(\n            data_path=dfile_path,\n            tokenizer=tokenizer,\n            label2id=label2id,\n            header_names=header,\n            max_seq_len=max_seq_len\n        )\n        try:\n            torch.save(features, cached_path)\n        except Exception:\n            pass\n\n    return NerDataset(features)\n',
    'src/helper.py': 'import os\nimport random\nimport logging\nimport argparse\nfrom typing import List, Optional\n\nimport numpy as np\nimport torch\n\nif hasattr(torch.serialization, \'add_safe_globals\'):\n    try:\n        torch.serialization.add_safe_globals([argparse.Namespace])\n    except Exception:\n        pass\n\n\ndef safe_torch_load(f, map_location=None, **kwargs):\n    """Load checkpoints on PyTorch versions with either weights_only default."""\n    try:\n        return torch.load(f, map_location=map_location, weights_only=False, **kwargs)\n    except TypeError:\n        return torch.load(f, map_location=map_location, **kwargs)\n\n\ndef set_random_seed(seed: int = 42):\n    os.environ[\'PYTHONHASHSEED\'] = str(seed)\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n    if hasattr(torch.backends, \'cudnn\'):\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n\n\ndef seed_worker(worker_id: int):\n    worker_seed = torch.initial_seed() % (2 ** 32)\n    random.seed(worker_seed)\n    np.random.seed(worker_seed)\n    torch.manual_seed(worker_seed)\n\n\ndef make_generator(seed: int):\n    generator = torch.Generator()\n    generator.manual_seed(seed)\n    return generator\n\n\ndef init_logger(log_file: Optional[str] = None, log_file_level=logging.INFO):\n    log_format = logging.Formatter(\'[%(asctime)s] [%(levelname)s] %(message)s\', datefmt=\'%Y-%m-%d %H:%M:%S\')\n    logger = logging.getLogger(\'baseline_supcon\')\n    logger.setLevel(logging.INFO)\n    if not logger.handlers:\n        console_handler = logging.StreamHandler()\n        console_handler.setFormatter(log_format)\n        logger.addHandler(console_handler)\n    if log_file:\n        try:\n            os.makedirs(\'./logs\', exist_ok=True)\n            log_path = os.path.join(\'./logs\', log_file)\n            file_handler = logging.FileHandler(log_path, encoding=\'utf-8\')\n            file_handler.setLevel(log_file_level)\n            file_handler.setFormatter(log_format)\n            logger.addHandler(file_handler)\n        except Exception as exc:\n            logger.warning(f\'Could not initialize file logging: {exc}\')\n    return logger\n\n\ndef get_total_model_parameters(model: torch.nn.Module):\n    total_params = sum(parameter.numel() for parameter in model.parameters())\n    trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)\n    return total_params, trainable_params\n\n\ndef plot_confusion_matrix(golds: List[int], preds: List[int], classes: List[str],\n                          output_dir: str = \'./outputs\', title: str = \'Confusion Matrix\',\n                          normalize: bool = True):\n    try:\n        import matplotlib\n        matplotlib.use(\'Agg\')\n        import matplotlib.pyplot as plt\n        from sklearn.metrics import confusion_matrix\n        matrix = confusion_matrix(golds, preds, labels=list(range(len(classes))))\n        if normalize:\n            matrix = matrix.astype(\'float\') / (matrix.sum(axis=1, keepdims=True) + 1e-12)\n        plt.figure(figsize=(10, 8))\n        plt.imshow(matrix, interpolation=\'nearest\', cmap=plt.cm.Blues)\n        plt.title(title)\n        plt.colorbar()\n        positions = np.arange(len(classes))\n        plt.xticks(positions, classes, rotation=45, ha=\'right\')\n        plt.yticks(positions, classes)\n        threshold = matrix.max() / 2.\n        for row in range(matrix.shape[0]):\n            for column in range(matrix.shape[1]):\n                plt.text(column, row, format(matrix[row, column], \'.2f\' if normalize else \'d\'),\n                         ha=\'center\', color=\'white\' if matrix[row, column] > threshold else \'black\')\n        plt.ylabel(\'True label\')\n        plt.xlabel(\'Predicted label\')\n        plt.tight_layout()\n        os.makedirs(output_dir, exist_ok=True)\n        plt.savefig(os.path.join(output_dir, \'confusion_matrix.png\'), dpi=300)\n        plt.close()\n    except Exception as exc:\n        logging.getLogger(\'baseline_supcon\').warning(f\'Could not save confusion matrix: {exc}\')\n',
    'src/processor.py': 'import os\nfrom pathlib import Path\nfrom typing import List, Union\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom tqdm import tqdm\n\n\nclass NerFeatures(object):\n    def __init__(self, input_ids, token_type_ids=None, attention_mask=None, valid_ids=None, labels=None, label_masks=None, tokens=None, **kwargs):\n        self.input_ids = torch.as_tensor(input_ids, dtype=torch.long)\n        self.token_type_ids = (torch.as_tensor(token_type_ids, dtype=torch.long)\n                               if token_type_ids is not None\n                               else torch.zeros_like(self.input_ids))\n        self.attention_mask = (torch.as_tensor(attention_mask, dtype=torch.long)\n                               if attention_mask is not None\n                               else torch.ones_like(self.input_ids))\n        self.valid_ids = torch.as_tensor(valid_ids, dtype=torch.long) if valid_ids is not None else None\n        self.labels = torch.as_tensor(labels, dtype=torch.long) if labels is not None else None\n        self.label_masks = torch.as_tensor(label_masks, dtype=torch.long) if label_masks is not None else None\n        self.tokens = list(tokens) if tokens is not None else []\n\n\ndef _read_conll_frame(path, header_names):\n    """Doc CoNLL thu cong: giu nguyen moi token (ke ca \'NA\', \'null\', \'nan\' ma pandas coi la NaN);\n    dong trong -> hang NaN (ranh gioi cau)."""\n    rows = []\n    with open(path, encoding=\'utf-8-sig\') as stream:\n        for line in stream:\n            fields = line.split()\n            if not fields:\n                rows.append([None] * len(header_names))\n            else:\n                fields = fields[:len(header_names)]\n                rows.append(fields + [None] * (len(header_names) - len(fields)))\n    return pd.DataFrame(rows, columns=header_names, dtype=object)\n\n\ndef _tag_column(data):\n    """File CoNLL 5 cột: nhãn ở cột 4 (\'ner\'); file 2 cột: nhãn nằm ở cột 2 (\'pos\')."""\n    return \'ner\' if data[\'ner\'].notna().any() else \'pos\'\n\n\ndef _count_conll_blocks(path):\n    count, inside = 0, False\n    with open(path, encoding=\'utf-8-sig\') as stream:\n        for line in stream:\n            if line.strip():\n                inside = True\n            elif inside:\n                count += 1\n                inside = False\n    return count + int(inside)\n\n\ndef _read_sentences(data_path: Union[str, os.PathLike], label2id: List[str], header_names: List[str]):\n    data = _read_conll_frame(data_path, header_names)\n    tag_column = _tag_column(data)\n    sentences = []\n    tokens, tag_ids = [], []\n    for _, row in data.iterrows():\n        token_val = row.get(\'token\')\n        if pd.isna(token_val) or not str(token_val).strip():\n            if tokens:\n                sentences.append((tokens, tag_ids))\n                tokens, tag_ids = [], []\n            continue\n        tokens.append(str(token_val).strip().replace(\' \', \'_\'))\n        tag = str(row.get(tag_column, \'O\')).strip()\n        tag_ids.append(label2id.index(tag) if tag in label2id else 0)\n    if tokens:\n        sentences.append((tokens, tag_ids))\n    return sentences\n\n\ndef _fast_encoding(tokenizer, words, tags, max_seq_len):\n    encoding = tokenizer(words, is_split_into_words=True, add_special_tokens=True,\n                         padding=\'max_length\', truncation=True, max_length=max_seq_len)\n    word_ids = encoding.word_ids()\n    positions = {}\n    for position, word_id in enumerate(word_ids):\n        if word_id is not None and word_id < len(words):\n            positions.setdefault(word_id, position)\n    kept_words = [word_id for word_id in range(len(words)) if word_id in positions]\n    return (encoding, [positions[word_id] for word_id in kept_words],\n            [tags[word_id] for word_id in kept_words], [words[word_id] for word_id in kept_words])\n\n\ndef _slow_encoding(tokenizer, words, tags, max_seq_len):\n    content_ids, content_word_ids = [], []\n    for word_id, word in enumerate(words):\n        pieces = tokenizer.tokenize(word)\n        piece_ids = tokenizer.convert_tokens_to_ids(pieces)\n        if isinstance(piece_ids, int):\n            piece_ids = [piece_ids]\n        content_ids.extend(piece_ids)\n        content_word_ids.extend([word_id] * len(piece_ids))\n\n    max_content = max_seq_len - tokenizer.num_special_tokens_to_add(pair=False)\n    content_ids = content_ids[:max_content]\n    content_word_ids = content_word_ids[:max_content]\n    input_ids = tokenizer.build_inputs_with_special_tokens(content_ids)\n    special_mask = tokenizer.get_special_tokens_mask(content_ids, already_has_special_tokens=False)\n    sequence_word_ids = []\n    content_index = 0\n    for is_special in special_mask:\n        if is_special:\n            sequence_word_ids.append(None)\n        else:\n            sequence_word_ids.append(content_word_ids[content_index])\n            content_index += 1\n\n    positions = {}\n    for position, word_id in enumerate(sequence_word_ids):\n        if word_id is not None:\n            positions.setdefault(word_id, position)\n    kept_words = [word_id for word_id in range(len(words)) if word_id in positions]\n    kept_tags = [tags[word_id] for word_id in kept_words]\n    attention_mask = [1] * len(input_ids)\n    if hasattr(tokenizer, \'create_token_type_ids_from_sequences\'):\n        token_type_ids = tokenizer.create_token_type_ids_from_sequences(content_ids)\n    else:\n        token_type_ids = [0] * len(input_ids)\n\n    pad_count = max_seq_len - len(input_ids)\n    if pad_count < 0:\n        input_ids = input_ids[:max_seq_len]\n        attention_mask = attention_mask[:max_seq_len]\n        token_type_ids = token_type_ids[:max_seq_len]\n        positions = {k: v for k, v in positions.items() if v < max_seq_len}\n        kept_words = [word_id for word_id in kept_words if word_id in positions]\n        kept_tags = [tags[word_id] for word_id in kept_words]\n    elif tokenizer.padding_side == \'left\':\n        input_ids = [tokenizer.pad_token_id] * pad_count + input_ids\n        attention_mask = [0] * pad_count + attention_mask\n        token_type_ids = [0] * pad_count + token_type_ids\n        positions = {k: v + pad_count for k, v in positions.items()}\n    else:\n        input_ids += [tokenizer.pad_token_id] * pad_count\n        attention_mask += [0] * pad_count\n        token_type_ids += [0] * pad_count\n\n    encoding = {\n        \'input_ids\': input_ids,\n        \'attention_mask\': attention_mask,\n        \'token_type_ids\': token_type_ids,\n    }\n    return (encoding, [positions[word_id] for word_id in kept_words], kept_tags,\n            [words[word_id] for word_id in kept_words])\n\n\ndef _uses_phobert_legacy_alignment(tokenizer):\n    """Choose the original first-subword/@@ path for PhoBERT-compatible BPE."""\n    model_name = str(getattr(tokenizer, \'name_or_path\', \'\')).lower().replace(\'\\\\\', \'/\').rstrip(\'/\')\n    model_id = model_name.rsplit(\'/\', 1)[-1]\n    if model_id in {\'phobert-base\', \'distilphobert\'}:\n        return True\n    if model_id in {\'xlm-roberta-base\', \'electra-base-vn\', \'bamibert\'}:\n        return False\n\n    get_vocab = getattr(tokenizer, \'get_vocab\', None)\n    if callable(get_vocab):\n        vocabulary = get_vocab()\n        return any(str(token).endswith(\'@@\') for token in vocabulary)\n    return False\n\n\ndef _convert_phobert_legacy_features(data_path: Union[str, os.PathLike],\n                                           tokenizer,\n                                           label2id: List[str],\n                                           header_names: List[str],\n                                           max_seq_len: int = 256) -> List[NerFeatures]:\n    """\n    Process Vietnamese word-segmented dataset into PhoBERT features with subword alignment.\n    Aligns BPE subwords back to original words via valid_ids.\n    Compatible with all pandas and PyTorch versions.\n    """\n    features = []\n    tokens = []\n    tag_ids = []\n\n    # Đọc dữ liệu CoNLL (dấu cách hoặc tab)\n    data = _read_conll_frame(data_path, header_names)\n    \n    # KHÔNG ffill: dòng trống (NaN) là ranh giới câu; ffill sẽ gộp cả file thành 1 câu.\n    tag_column = _tag_column(data)\n\n    total_rows = len(data)\n    for row_idx, row in tqdm(data.iterrows(), total=total_rows, desc=f"Processing {os.path.basename(data_path)}..."):\n        token_val = row.get(\'token\', None) if hasattr(row, \'get\') else getattr(row, \'token\', None)\n        ner_val = row.get(tag_column, None) if hasattr(row, \'get\') else getattr(row, tag_column, None)\n\n        if pd.notna(token_val):  # token \'nan\'/\'NA\' van la token hop le; chi hang None (dong trong) la ranh gioi cau\n            tokens.append(str(token_val).strip().replace(\' \', \'_\'))\n            ner_tag = str(ner_val).strip()\n            if ner_tag in label2id:\n                tag_ids.append(label2id.index(ner_tag))\n            else:\n                tag_ids.append(0)  # default \'O\'\n            if row_idx < total_rows - 1:\n                continue\n\n        seq_len = len(tokens)\n        if seq_len == 0:\n            continue\n\n        sentence = \' \'.join(tokens)\n        encoding = tokenizer(sentence,\n                             padding=\'max_length\',\n                             truncation=True,\n                             max_length=max_seq_len)\n\n        subwords = tokenizer.tokenize(sentence)\n        valid_ids = np.zeros(len(encoding.input_ids), dtype=int)\n        label_masks = np.zeros(len(encoding.input_ids), dtype=int)\n\n        i = 1\n        for idx, subword in enumerate(subwords[:max_seq_len - 2]):\n            if idx != 0 and subwords[idx - 1].endswith("@@"):\n                continue\n            valid_ids[i - 1] = idx + 1\n            i += 1\n\n        if max_seq_len >= seq_len:\n            label_padding_size = (max_seq_len - seq_len)\n            label_masks[:seq_len] = [1] * seq_len\n            tag_ids.extend([0] * label_padding_size)\n        else:\n            tag_ids = tag_ids[:max_seq_len]\n            label_masks[:-2] = [1] * (max_seq_len - 2)\n            tag_ids[-2:] = [0] * 2\n\n        items = {key: val for key, val in encoding.items()}\n        if \'token_type_ids\' not in items:\n            items[\'token_type_ids\'] = [0] * max_seq_len\n        items[\'labels\'] = tag_ids\n        items[\'valid_ids\'] = valid_ids\n        items[\'label_masks\'] = label_masks\n        items[\'tokens\'] = tokens.copy()\n\n        features.append(NerFeatures(**items))\n\n        tokens = []\n        tag_ids = []\n\n    return features\n\ndef _convert_features_impl(data_path: Union[str, os.PathLike],\n                                           tokenizer,\n                                           label2id: List[str],\n                                           header_names: List[str],\n                                           max_seq_len: int = 256) -> List[NerFeatures]:\n    """Encode word-segmented NER data and align each label to its first subword."""\n    if _uses_phobert_legacy_alignment(tokenizer):\n        return _convert_phobert_legacy_features(\n            data_path, tokenizer, label2id, header_names, max_seq_len\n        )\n\n    features = []\n    sentences = _read_sentences(data_path, label2id, header_names)\n    for words, tags in tqdm(sentences, desc=f\'Processing {os.path.basename(data_path)}...\'):\n        if getattr(tokenizer, \'is_fast\', False):\n            encoding, positions, kept_tags, kept_tokens = _fast_encoding(tokenizer, words, tags, max_seq_len)\n        else:\n            encoding, positions, kept_tags, kept_tokens = _slow_encoding(tokenizer, words, tags, max_seq_len)\n        input_ids = list(encoding[\'input_ids\'])\n        attention_mask = list(encoding.get(\'attention_mask\', [1] * len(input_ids)))\n        token_type_ids = list(encoding.get(\'token_type_ids\', [0] * len(input_ids)))\n        if len(input_ids) != max_seq_len:\n            raise ValueError(f\'Tokenizer returned {len(input_ids)} tokens; expected max_seq_len={max_seq_len}\')\n        valid_ids = np.zeros(max_seq_len, dtype=np.int64)\n        label_masks = np.zeros(max_seq_len, dtype=np.int64)\n        labels = np.zeros(max_seq_len, dtype=np.int64)\n        word_count = min(len(positions), max_seq_len)\n        valid_ids[:word_count] = positions[:word_count]\n        label_masks[:word_count] = 1\n        labels[:word_count] = kept_tags[:word_count]\n        features.append(NerFeatures(input_ids=input_ids, token_type_ids=token_type_ids,\n                                    attention_mask=attention_mask, valid_ids=valid_ids,\n                                    labels=labels, label_masks=label_masks,\n                                    tokens=kept_tokens[:word_count]))\n    return features\n\n\ndef convert_word_segment_examples_features(data_path: Union[str, os.PathLike],\n                                           tokenizer,\n                                           label2id: List[str],\n                                           header_names: List[str],\n                                           max_seq_len: int = 256) -> List[NerFeatures]:\n    features = _convert_features_impl(data_path, tokenizer, label2id, header_names, max_seq_len)\n    blocks = _count_conll_blocks(data_path)\n    if len(features) != blocks:\n        raise ValueError(f\'{data_path}: tao {len(features)} feature nhung file co {blocks} cau; kiem tra tach cau/dinh dang cot.\')\n    entity_tokens = sum(int(((f.labels != 0) & (f.label_masks != 0)).sum()) for f in features)\n    if blocks >= 100 and entity_tokens == 0:\n        raise ValueError(f\'{data_path}: khong co nhan thuc the nao; kiem tra cot nhan.\')\n    return features\n',
    'src/trainer.py': 'import contextlib\nimport datetime\nimport io\nimport json\nimport logging\nimport math\nimport os\nimport time\nfrom importlib.metadata import PackageNotFoundError, version\nfrom pathlib import Path\nfrom typing import Tuple\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nfrom sklearn.metrics import (accuracy_score, classification_report, f1_score,\n                             precision_score, recall_score)\nfrom torch.utils.data import DataLoader, RandomSampler, SequentialSampler\nfrom transformers import AutoTokenizer, get_linear_schedule_with_warmup\n\nfrom .constant import LOGGER, LABEL2ID_PAP_NER, LABEL_MAPPING\nfrom .conlleval import evaluate\nfrom .dataset import build_dataset\nfrom .helper import (make_generator, plot_confusion_matrix, safe_torch_load,\n                     seed_worker, set_random_seed)\nfrom .models import TokenEncoderLinearProbe, TokenEncoderSupCon\n\n\ndef _get_amp_context(use_amp: bool, device: torch.device):\n    if use_amp and device.type == \'cuda\':\n        return torch.autocast(device_type=\'cuda\', dtype=torch.float16)\n    return contextlib.nullcontext()\n\n\ndef _get_grad_scaler(use_amp: bool, device: torch.device):\n    if use_amp and device.type == \'cuda\':\n        try:\n            return torch.amp.GradScaler(\'cuda\')\n        except (AttributeError, TypeError):\n            return torch.cuda.amp.GradScaler()\n    return None\n\n\ndef _write_json(path: Path, content):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(content, ensure_ascii=False, indent=2, default=str), encoding=\'utf-8\')\n\n\ndef _make_loader(dataset, batch_size: int, shuffle: bool, seed: int, device: torch.device, workers: int):\n    sampler_generator = make_generator(seed)\n    loader_generator = make_generator(seed + 1)\n    sampler = RandomSampler(dataset, generator=sampler_generator) if shuffle else SequentialSampler(dataset)\n    return DataLoader(dataset, sampler=sampler, batch_size=batch_size, num_workers=workers,\n                      pin_memory=device.type == \'cuda\', worker_init_fn=seed_worker,\n                      generator=loader_generator, persistent_workers=workers > 0)\n\n\ndef _effective_batch_groups(iterator, accumulation_steps: int):\n    group = []\n    for batch in iterator:\n        group.append(batch)\n        if len(group) == accumulation_steps:\n            yield group\n            group = []\n    if group:\n        yield group\n\n\ndef _move_batch(batch, device):\n    return {key: value.to(device, non_blocking=True) if isinstance(value, torch.Tensor) else value\n            for key, value in batch.items()}\n\n\ndef _entity_mask(batch):\n    return (batch[\'label_masks\'] != 0) & (batch[\'labels\'] != 0)\n\n\ndef _capture_rng(device):\n    cpu_state = torch.get_rng_state()\n    cuda_state = torch.cuda.get_rng_state(device) if device.type == \'cuda\' else None\n    return cpu_state, cuda_state\n\n\ndef _restore_rng(state, device):\n    cpu_state, cuda_state = state\n    torch.set_rng_state(cpu_state)\n    if cuda_state is not None:\n        torch.cuda.set_rng_state(cuda_state, device)\n\n\ndef _stage1_group_backward(model, group, device, use_amp, scaler, optimizer, max_grad_norm):\n    if len(group) == 1:\n        batch = _move_batch(group[0], device)\n        with _get_amp_context(use_amp, device):\n            loss = model(**batch).loss\n        loss_value = float(loss.detach().item())\n        if loss_value > 0:\n            _backward_and_step(loss, model, optimizer, scaler, max_grad_norm)\n        return loss_value\n\n    rng_states, entity_features, entity_tags, entity_counts, masks = [], [], [], [], []\n    for raw_batch in group:\n        batch = _move_batch(raw_batch, device)\n        rng_states.append(_capture_rng(device))\n        with torch.no_grad(), _get_amp_context(use_amp, device):\n            word_representations = model.encode_word_representations(**batch)\n        entity_mask = _entity_mask(batch)\n        masks.append(entity_mask.detach())\n        selected = word_representations[entity_mask].detach()\n        entity_features.append(selected.cpu())\n        entity_tags.append(batch[\'labels\'][entity_mask].detach().cpu())\n        entity_counts.append(int(selected.size(0)))\n    after_forward_rng = _capture_rng(device)\n    all_features = torch.cat(entity_features, dim=0).to(device).detach().requires_grad_(True)\n    all_tags = torch.cat(entity_tags, dim=0).to(device)\n    if all_features.size(0) < 2:\n        return 0.0\n\n    with _get_amp_context(use_amp, device):\n        projected = model.projection_head(all_features)\n        loss = model.supcon_loss(projected, all_tags)\n    loss_value = float(loss.detach().item())\n    if loss_value <= 0:\n        return loss_value\n    (scaler.scale(loss) if scaler is not None else loss).backward()\n    feature_gradients = all_features.grad.detach().split(entity_counts)\n    offset = 0\n    try:\n        for index, (raw_batch, rng_state, mask, count) in enumerate(zip(group, rng_states, masks, entity_counts)):\n            _restore_rng(rng_state, device)\n            if count == 0:\n                continue\n            batch = _move_batch(raw_batch, device)\n            with _get_amp_context(use_amp, device):\n                word_representations = model.encode_word_representations(**batch)\n                selected = word_representations[mask]\n                gradient = feature_gradients[index].to(device)\n                surrogate = (selected.float() * gradient.float()).sum()\n            surrogate.backward()\n    finally:\n        _restore_rng(after_forward_rng, device)\n    _optimizer_step(model, optimizer, scaler, max_grad_norm)\n    return loss_value\n\n\ndef _backward_and_step(loss, model, optimizer, scaler, max_grad_norm):\n    (scaler.scale(loss) if scaler is not None else loss).backward()\n    _optimizer_step(model, optimizer, scaler, max_grad_norm)\n\n\ndef _optimizer_step(model, optimizer, scaler, max_grad_norm):\n    if scaler is not None:\n        scaler.unscale_(optimizer)\n    nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)\n    if scaler is None:\n        optimizer.step()\n    else:\n        scaler.step(optimizer)\n        scaler.update()\n\n\ndef _stage1_group_loss(model, group, device, use_amp):\n    features, tags = [], []\n    with torch.no_grad():\n        for raw_batch in group:\n            batch = _move_batch(raw_batch, device)\n            with _get_amp_context(use_amp, device):\n                word_representations = model.encode_word_representations(**batch)\n            mask = _entity_mask(batch)\n            features.append(word_representations[mask])\n            tags.append(batch[\'labels\'][mask])\n        if not features:\n            return 0.0\n        all_features = torch.cat(features, dim=0)\n        if all_features.size(0) < 2:\n            return 0.0\n        all_tags = torch.cat(tags, dim=0)\n        with _get_amp_context(use_amp, device):\n            projected = model.projection_head(all_features)\n            loss = model.supcon_loss(projected, all_tags)\n        return float(loss.item())\n\n\ndef train_stage1_supcon(args):\n    set_random_seed(args.seed)\n    output_dir = Path(args.output_dir).resolve()\n    output_dir.mkdir(parents=True, exist_ok=True)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() and not getattr(args, \'no_cuda\', False) else \'cpu\')\n    micro_batch_size = int(getattr(args, \'micro_batch_size\', args.batch_size))\n    accumulation_steps = args.batch_size // micro_batch_size\n    if micro_batch_size * accumulation_steps != args.batch_size:\n        raise ValueError(\'micro_batch_size × accumulation_steps must equal the configured effective batch size\')\n    LOGGER.info(f\'Stage 1 seed={args.seed}, model={args.model_name_or_path}, effective_batch={args.batch_size}, micro={micro_batch_size}, accum={accumulation_steps}, device={device}\')\n    tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path, use_fast=getattr(args, \'use_fast_tokenizer\', False))\n    train_dataset = build_dataset(args.data_dir, tokenizer, LABEL2ID_PAP_NER,\n                                  LABEL_MAPPING[\'pap_ner\'][\'header\'], \'train\', args.max_seq_length,\n                                  getattr(args, \'overwrite_data\', False))\n    dev_dataset = build_dataset(args.data_dir, tokenizer, LABEL2ID_PAP_NER,\n                                LABEL_MAPPING[\'pap_ner\'][\'header\'], \'dev\', args.max_seq_length,\n                                getattr(args, \'overwrite_data\', False))\n    train_loader = _make_loader(train_dataset, micro_batch_size, True, args.seed, device, args.num_workers)\n    dev_loader = _make_loader(dev_dataset, micro_batch_size, False, args.seed + 10_000, device, args.num_workers)\n    model = TokenEncoderSupCon(args.model_name_or_path, args.projection_dim, args.supcon_temp, 0).to(device)\n    optimizer = torch.optim.AdamW([\n        {\'params\': model.encoder.parameters(), \'lr\': args.learning_rate},\n        {\'params\': model.projection_head.parameters(), \'lr\': args.head_learning_rate},\n    ], weight_decay=args.weight_decay)\n    updates_per_epoch = math.ceil(len(train_dataset) / args.batch_size)\n    total_steps = updates_per_epoch * args.epochs\n    warmup_steps = int(total_steps * args.warmup_proportion)\n    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)\n    use_amp = bool(args.fp16 and device.type == \'cuda\')\n    scaler = _get_grad_scaler(use_amp, device)\n    best_dev_loss, best_epoch, early_stop_count = float(\'inf\'), 0, 0\n    best_model_path = output_dir / \'best_encoder.pt\'\n    history = {\'seed\': args.seed, \'stage1\': []}\n    stage_start = time.time()\n\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        epoch_start = time.time()\n        train_losses = []\n        for group in _effective_batch_groups(train_loader, accumulation_steps):\n            optimizer.zero_grad(set_to_none=True)\n            loss_value = _stage1_group_backward(model, group, device, use_amp, scaler,\n                                                optimizer, args.max_grad_norm)\n            if math.isfinite(loss_value):\n                train_losses.append(loss_value)\n            scheduler.step()\n        model.eval()\n        dev_losses = [_stage1_group_loss(model, group, device, use_amp)\n                      for group in _effective_batch_groups(dev_loader, accumulation_steps)]\n        average_train_loss = float(np.mean(train_losses)) if train_losses else 0.0\n        average_dev_loss = float(np.mean(dev_losses)) if dev_losses else 0.0\n        duration = time.time() - epoch_start\n        LOGGER.info(f\'[STAGE 1 - Epoch {epoch:02d}/{args.epochs:02d}] Train Loss: {average_train_loss:.4f} | Dev Loss: {average_dev_loss:.4f} | Time: {datetime.timedelta(seconds=int(duration))}\')\n        history[\'stage1\'].append({\'epoch\': epoch, \'train_loss\': average_train_loss,\n                                  \'val_loss\': average_dev_loss, \'duration_seconds\': duration})\n        if average_dev_loss < best_dev_loss:\n            best_dev_loss, best_epoch, early_stop_count = average_dev_loss, epoch, 0\n            torch.save({\'epoch\': epoch, \'encoder_state_dict\': model.encoder.state_dict(),\n                        \'projection_head_state_dict\': model.projection_head.state_dict(),\n                        \'dev_loss\': average_dev_loss, \'args\': vars(args)}, best_model_path)\n        else:\n            early_stop_count += 1\n            if early_stop_count >= args.early_stop:\n                LOGGER.info(f\'Stage 1 early stopping at epoch {epoch}.\')\n                break\n        history.update(best_epoch=best_epoch, epochs_run=len(history[\'stage1\']),\n                       total_duration_seconds=time.time() - stage_start)\n        _write_json(output_dir / \'history.json\', history)\n\n    history.update(best_epoch=best_epoch, epochs_run=len(history[\'stage1\']),\n                   total_duration_seconds=time.time() - stage_start)\n    _write_json(output_dir / \'history.json\', history)\n    if best_epoch == 0:\n        raise RuntimeError(\'Stage 1 produced no finite validation loss; no encoder checkpoint was saved.\')\n    return str(best_model_path)\n\n\ndef _stage2_batch_backward(model, raw_batch, device, use_amp, scaler, optimizer):\n    """Run the unchanged one-batch Stage 2 optimizer step."""\n    batch = _move_batch(raw_batch, device)\n    optimizer.zero_grad(set_to_none=True)\n    with _get_amp_context(use_amp, device):\n        outputs = model(**batch)\n        loss = outputs.loss\n    if scaler is None:\n        loss.backward()\n        optimizer.step()\n    else:\n        scaler.scale(loss).backward()\n        scaler.step(optimizer)\n        scaler.update()\n    return float(loss.detach().item())\n\n\ndef _stage2_group_backward(model, group, device, use_amp, scaler, optimizer):\n    active_count = sum(int((batch[\'label_masks\'] != 0).sum().item()) for batch in group)\n    if active_count == 0:\n        return 0.0\n    accumulated_loss = 0.0\n    for raw_batch in group:\n        batch = _move_batch(raw_batch, device)\n        count = int((batch[\'label_masks\'] != 0).sum().item())\n        with _get_amp_context(use_amp, device):\n            outputs = model(**batch)\n            loss = outputs.loss\n        # outputs.loss is a mean over active tokens in this micro-batch.\n        # Weight by its active-token count so the accumulated gradient equals\n        # one mean-token CE loss over the configured effective batch.\n        normalized_loss = loss * (count / active_count)\n        (scaler.scale(normalized_loss) if scaler is not None else normalized_loss).backward()\n        accumulated_loss += float(loss.detach().item()) * count / active_count\n    if scaler is None:\n        optimizer.step()\n    else:\n        scaler.step(optimizer)\n        scaler.update()\n    return accumulated_loss\n\n\ndef train_stage2_linear(args):\n    set_random_seed(args.seed)\n    output_dir = Path(args.output_dir).resolve()\n    output_dir.mkdir(parents=True, exist_ok=True)\n    seed_dir = output_dir.parent\n    device = torch.device(\'cuda\' if torch.cuda.is_available() and not getattr(args, \'no_cuda\', False) else \'cpu\')\n    micro_batch_size = int(getattr(args, \'micro_batch_size\', args.batch_size))\n    accumulation_steps = args.batch_size // micro_batch_size\n    if micro_batch_size * accumulation_steps != args.batch_size:\n        raise ValueError(\'micro_batch_size × accumulation_steps must equal the configured effective batch size\')\n    LOGGER.info(f\'Stage 2 seed={args.seed}, model={args.model_name_or_path}, effective_batch={args.batch_size}, micro={micro_batch_size}, accum={accumulation_steps}, device={device}\')\n    tokenizer = AutoTokenizer.from_pretrained(args.model_name_or_path, use_fast=getattr(args, \'use_fast_tokenizer\', False))\n    datasets = {\n        split: build_dataset(args.data_dir, tokenizer, LABEL2ID_PAP_NER,\n                             LABEL_MAPPING[\'pap_ner\'][\'header\'], split, args.max_seq_length,\n                             getattr(args, \'overwrite_data\', False))\n        for split in (\'train\', \'dev\', \'test\')\n    }\n    train_loader = _make_loader(datasets[\'train\'], micro_batch_size, True, args.seed, device, args.num_workers)\n    dev_loader = _make_loader(datasets[\'dev\'], micro_batch_size, False, args.seed + 20_000, device, args.num_workers)\n    test_loader = _make_loader(datasets[\'test\'], micro_batch_size, False, args.seed + 30_000, device, args.num_workers)\n    model = TokenEncoderLinearProbe(args.model_name_or_path, len(LABEL2ID_PAP_NER), args.freeze_backbone).to(device)\n    checkpoint = safe_torch_load(args.pretrained_encoder_path, map_location=device)\n    model.encoder.load_state_dict(checkpoint[\'encoder_state_dict\'])\n    optimizer = torch.optim.Adam(model.classifier.parameters(), lr=args.classifier_learning_rate)\n    use_amp = bool(args.fp16 and device.type == \'cuda\')\n    scaler = _get_grad_scaler(use_amp, device)\n    best_dev_f1, best_epoch, early_stop_count = 0.0, 0, 0\n    best_model_path = output_dir / \'best_linear_probe.pt\'\n    history = {\'seed\': args.seed, \'stage2\': []}\n    stage_start = time.time()\n\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        model.encoder.eval()\n        epoch_start = time.time()\n        train_losses = []\n        if micro_batch_size == args.batch_size:\n            for batch in train_loader:\n                train_losses.append(_stage2_batch_backward(\n                    model, batch, device, use_amp, scaler, optimizer\n                ))\n        else:\n            for group in _effective_batch_groups(train_loader, accumulation_steps):\n                optimizer.zero_grad(set_to_none=True)\n                train_losses.append(_stage2_group_backward(\n                    model, group, device, use_amp, scaler, optimizer\n                ))\n        dev_precision, dev_recall, dev_f1 = evaluate_linear_probe(model, dev_loader, device=device, is_test=False)\n        duration = time.time() - epoch_start\n        average_train_loss = float(np.mean(train_losses)) if train_losses else 0.0\n        LOGGER.info(f\'[STAGE 2 - Epoch {epoch:02d}/{args.epochs:02d}] Train Loss: {average_train_loss:.4f} | Dev Strict P: {dev_precision:.4f} | R: {dev_recall:.4f} | F1: {dev_f1:.4f} | Time: {datetime.timedelta(seconds=int(duration))}\')\n        history[\'stage2\'].append({\'epoch\': epoch, \'train_loss\': average_train_loss,\n                                  \'dev_precision\': dev_precision, \'dev_recall\': dev_recall,\n                                  \'dev_f1\': dev_f1, \'duration_seconds\': duration})\n        if dev_f1 > best_dev_f1:\n            best_dev_f1, best_epoch, early_stop_count = dev_f1, epoch, 0\n            torch.save({\'epoch\': epoch, \'model_state_dict\': model.state_dict(),\n                        \'classes\': LABEL2ID_PAP_NER, \'best_dev_f1\': best_dev_f1,\n                        \'args\': vars(args)}, best_model_path)\n        else:\n            early_stop_count += 1\n            if early_stop_count >= args.early_stop:\n                LOGGER.info(f\'Stage 2 early stopping at epoch {epoch}.\')\n                break\n        history.update(best_epoch=best_epoch, epochs_run=len(history[\'stage2\']),\n                       total_duration_seconds=time.time() - stage_start)\n        _write_json(output_dir / \'history.json\', history)\n\n    history.update(best_epoch=best_epoch, epochs_run=len(history[\'stage2\']),\n                   total_duration_seconds=time.time() - stage_start)\n    _write_json(output_dir / \'history.json\', history)\n    best_checkpoint = safe_torch_load(best_model_path, map_location=device)\n    model.load_state_dict(best_checkpoint[\'model_state_dict\'])\n    test_metrics = evaluate_linear_probe(model, test_loader, device=device, is_test=True,\n                                         output_dir=str(output_dir), seed=args.seed,\n                                         model_choice=getattr(args, \'model_choice\', \'\'),\n                                         model_name=args.model_name_or_path)\n    LOGGER.info(f\'Test Strict P={test_metrics[0]:.4f} R={test_metrics[1]:.4f} F1={test_metrics[2]:.4f}\')\n    return test_metrics\n\n\ndef _extract_entities(tags):\n    entities = []\n    start, entity_type = None, None\n    for index in range(len(tags) + 1):\n        tag = tags[index] if index < len(tags) else \'O\'\n        prefix, current_type = (\'O\', None) if tag == \'O\' else tag.split(\'-\', 1)\n        continuation = prefix == \'I\' and entity_type == current_type and start is not None\n        if start is not None and not continuation:\n            entities.append((entity_type, start, index - 1))\n            start, entity_type = None, None\n        if prefix == \'B\' or (prefix == \'I\' and not continuation):\n            start, entity_type = index, current_type\n    return entities\n\n\ndef _strict_entity_metrics(gold_sequences, pred_sequences, entity_types):\n    counts = {entity_type: {\'gold\': 0, \'pred\': 0, \'correct\': 0} for entity_type in entity_types}\n    total_gold = total_pred = total_correct = 0\n    for gold_tags, pred_tags in zip(gold_sequences, pred_sequences):\n        gold_entities = set(_extract_entities(gold_tags))\n        pred_entities = set(_extract_entities(pred_tags))\n        for entity_type in entity_types:\n            gold_typed = {entity for entity in gold_entities if entity[0] == entity_type}\n            pred_typed = {entity for entity in pred_entities if entity[0] == entity_type}\n            counts[entity_type][\'gold\'] += len(gold_typed)\n            counts[entity_type][\'pred\'] += len(pred_typed)\n            counts[entity_type][\'correct\'] += len(gold_typed & pred_typed)\n        total_gold += len(gold_entities)\n        total_pred += len(pred_entities)\n        total_correct += len(gold_entities & pred_entities)\n    results = {}\n    for entity_type, values in counts.items():\n        results[entity_type] = _prf(values[\'correct\'], values[\'pred\'], values[\'gold\'])\n    results[\'Micro\'] = _prf(total_correct, total_pred, total_gold)\n    return results\n\n\ndef _prf(correct, predicted, gold):\n    precision = correct / predicted if predicted else 0.0\n    recall = correct / gold if gold else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    return {\'precision\': precision, \'recall\': recall, \'f1\': f1}\n\n\ndef _nervaluate_strict_metrics(gold_sequences, pred_sequences, entity_types, fallback):\n    method, results = \'manual_exact_span_type\', fallback\n    try:\n        if version(\'nervaluate\') != \'1.2.1\':\n            raise RuntimeError(\'nervaluate version is not 1.2.1\')\n        try:\n            from nervaluate import Evaluator\n        except ImportError:\n            from nervaluate.evaluator import Evaluator\n        nervaluate_results = Evaluator(gold_sequences, pred_sequences, tags=entity_types, loader=\'list\').evaluate()\n        overall = nervaluate_results[\'overall\'][\'strict\']\n        by_entity = nervaluate_results[\'entities\']\n        candidate = {\'Micro\': {\'precision\': overall.precision, \'recall\': overall.recall, \'f1\': overall.f1}}\n        for entity_type in entity_types:\n            metric = by_entity[entity_type][\'strict\']\n            candidate[entity_type] = {\'precision\': metric.precision, \'recall\': metric.recall, \'f1\': metric.f1}\n        for key, values in fallback.items():\n            if not all(math.isclose(candidate[key][metric], values[metric], abs_tol=1e-8)\n                       for metric in (\'precision\', \'recall\', \'f1\')):\n                raise RuntimeError(f\'nervaluate strict metrics differ from exact span/type counts for {key}\')\n        method, results = \'nervaluate==1.2.1 (EvaluationResult attributes)\', candidate\n    except (ImportError, PackageNotFoundError, KeyError, TypeError, AttributeError, RuntimeError) as exc:\n        LOGGER.warning(f\'Using manual strict span/type calculation: {exc}\')\n    return results, method\n\n\ndef _bio_metrics(gold_ids, pred_ids):\n    labels = list(range(len(LABEL2ID_PAP_NER)))\n    non_o = [index for index, gold in enumerate(gold_ids) if gold != 0]\n    non_o_accuracy = accuracy_score([gold_ids[index] for index in non_o],\n                                    [pred_ids[index] for index in non_o]) if non_o else 0.0\n    return {\n        \'Accuracy không tính O (%)\': round(100 * non_o_accuracy, 2),\n        \'BIO Accuracy (%)\': round(100 * accuracy_score(gold_ids, pred_ids), 2),\n        \'BIO Macro Recall (%)\': round(100 * recall_score(gold_ids, pred_ids, labels=labels, average=\'macro\', zero_division=0), 2),\n        \'BIO Macro-F1 (%)\': round(100 * f1_score(gold_ids, pred_ids, labels=labels, average=\'macro\', zero_division=0), 2),\n        \'BIO Weighted Precision (%)\': round(100 * precision_score(gold_ids, pred_ids, labels=labels, average=\'weighted\', zero_division=0), 2),\n        \'BIO Weighted Recall (%)\': round(100 * recall_score(gold_ids, pred_ids, labels=labels, average=\'weighted\', zero_division=0), 2),\n        \'BIO Weighted-F1 (%)\': round(100 * f1_score(gold_ids, pred_ids, labels=labels, average=\'weighted\', zero_division=0), 2),\n    }\n\n\ndef _save_test_artifacts(seed_dir, gold_sequences, pred_sequences, gold_ids, pred_ids,\n                         metrics, seed, model_choice, model_name, report, advanced, method, token_sequences=None):\n    id2label = LABEL_MAPPING[\'pap_ner\'][\'id2label\']\n    gold_tags = [[id2label[tag] for tag in sentence] for sentence in gold_sequences]\n    pred_tags = [[id2label[tag] for tag in sentence] for sentence in pred_sequences]\n    prediction_data = {\'gold\': gold_tags, \'pred\': pred_tags}\n    if token_sequences is not None and len(token_sequences) == len(gold_tags):\n        prediction_data[\'tokens\'] = token_sequences\n    _write_json(seed_dir / \'test_predictions.json\', prediction_data)\n    (seed_dir / \'test_report.txt\').write_text(report, encoding=\'utf-8\')\n    _write_json(seed_dir / \'advanced_eval_table.json\', {\'method\': method, \'strict\': advanced})\n    advanced_rows = [{\'Tag\': tag, \'Strict_P\': values[\'precision\'] * 100,\n                      \'Strict_R\': values[\'recall\'] * 100, \'Strict_F1\': values[\'f1\'] * 100}\n                     for tag, values in advanced.items()]\n    pd.DataFrame(advanced_rows).to_csv(seed_dir / \'advanced_eval_table.csv\', index=False, encoding=\'utf-8-sig\')\n\n    bio_row = {\'Phiên bản\': model_choice, \'Model\': model_name, **_bio_metrics(gold_ids, pred_ids),\n               \'Ghi chú\': f\'Seed {seed}\'}\n    entity_row = {\'Phiên bản\': model_choice, \'Model\': model_name,\n                  \'CQ - P (%)\': round(100 * advanced[\'CQ\'][\'precision\'], 2),\n                  \'CQ - R (%)\': round(100 * advanced[\'CQ\'][\'recall\'], 2),\n                  \'VBPL - P (%)\': round(100 * advanced[\'VBPL\'][\'precision\'], 2),\n                  \'VBPL - R (%)\': round(100 * advanced[\'VBPL\'][\'recall\'], 2),\n                  \'VBPL - F1 (%)\': round(100 * advanced[\'VBPL\'][\'f1\'], 2),\n                  \'NG - P (%)\': round(100 * advanced[\'NG\'][\'precision\'], 2),\n                  \'NG - R (%)\': round(100 * advanced[\'NG\'][\'recall\'], 2),\n                  \'NG - F1 (%)\': round(100 * advanced[\'NG\'][\'f1\'], 2),\n                  \'SL - P (%)\': round(100 * advanced[\'SL\'][\'precision\'], 2),\n                  \'SL - R (%)\': round(100 * advanced[\'SL\'][\'recall\'], 2),\n                  \'SL - F1 (%)\': round(100 * advanced[\'SL\'][\'f1\'], 2),\n                  \'Overall - P (%)\': round(100 * advanced[\'Micro\'][\'precision\'], 2),\n                  \'Overall - R (%)\': round(100 * advanced[\'Micro\'][\'recall\'], 2),\n                  \'Overall - F1 (%)\': round(100 * advanced[\'Micro\'][\'f1\'], 2),\n                  \'Ghi chú\': f\'Seed {seed}\'}\n    pd.DataFrame([bio_row]).to_csv(seed_dir / \'paper_table1_bio.csv\', index=False, encoding=\'utf-8-sig\')\n    pd.DataFrame([entity_row]).to_csv(seed_dir / \'paper_table2_entity.csv\', index=False, encoding=\'utf-8-sig\')\n    _write_json(seed_dir / \'test_metrics.json\', {\n        \'seed\': seed, \'model_choice\': model_choice, \'model_name_or_path\': model_name,\n        \'strict_precision\': metrics[0], \'strict_recall\': metrics[1], \'strict_f1\': metrics[2],\n        \'bio\': bio_row, \'entity\': entity_row, \'entity_metric_method\': method,\n    })\n\n\ndef evaluate_linear_probe(model, iterator, device=\'cuda\', is_test: bool = False,\n                          output_dir: str = \'./outputs/stage2_linear\', seed=None,\n                          model_choice=\'\', model_name=\'\') -> Tuple[float, float, float]:\n    model.eval()\n    gold_sequences, pred_sequences, token_sequences = [], [], []\n    source_examples = getattr(getattr(iterator, \'dataset\', None), \'examples\', None)\n    sentence_index = 0\n    conll_log = io.StringIO()\n    with torch.no_grad():\n        for batch in iterator:\n            batch = _move_batch(batch, torch.device(device))\n            outputs = model(**batch)\n            active_mask = batch[\'label_masks\'] != 0\n            for row, predicted in enumerate(outputs.tags):\n                length = int(active_mask[row].sum().item())\n                gold_sequences.append(batch[\'labels\'][row, :length].cpu().tolist())\n                pred_sequences.append(list(predicted[:length]))\n                if source_examples is not None and sentence_index < len(source_examples):\n                    sentence_tokens = getattr(source_examples[sentence_index], \'tokens\', None)\n                    token_sequences.append(list(sentence_tokens[:length]) if sentence_tokens else [])\n                sentence_index += 1\n    if any(len(gold) != len(pred) for gold, pred in zip(gold_sequences, pred_sequences)):\n        raise ValueError(\'Gold/pred token alignment mismatch during evaluation.\')\n    eval_golds = [label for sentence in gold_sequences for label in sentence]\n    eval_preds = [label for sentence in pred_sequences for label in sentence]\n    id2label = LABEL_MAPPING[\'pap_ner\'][\'id2label\']\n    gold_str = [id2label[tag] for tag in eval_golds]\n    pred_str = [id2label[tag] for tag in eval_preds]\n    if is_test:\n        conll_handler = logging.StreamHandler(conll_log)\n        LOGGER.addHandler(conll_handler)\n        try:\n            prec, rec, f1 = evaluate(gold_str, pred_str, verbose=True)\n        finally:\n            LOGGER.removeHandler(conll_handler)\n    else:\n        prec, rec, f1 = evaluate(gold_str, pred_str, verbose=False)\n\n    if is_test:\n        report_labels = list(range(len(LABEL2ID_PAP_NER)))\n        report = classification_report(eval_golds, eval_preds, labels=report_labels,\n                                       target_names=LABEL2ID_PAP_NER, digits=4, zero_division=0)\n        LOGGER.info(\'\\n\' + report)\n        output_path = Path(output_dir).resolve()\n        seed_dir = output_path.parent\n        seed_dir.mkdir(parents=True, exist_ok=True)\n        plot_confusion_matrix(eval_golds, eval_preds, LABEL2ID_PAP_NER,\n                              output_dir=str(output_path),\n                              title=\'Normalized Confusion Matrix (Linear Probe Stage 2)\', normalize=True)\n        entity_types = [\'CQ\', \'ĐT\', \'VBPL\', \'NG\', \'SL\']\n        gold_tag_sequences = [[id2label[tag] for tag in sentence] for sentence in gold_sequences]\n        pred_tag_sequences = [[id2label[tag] for tag in sentence] for sentence in pred_sequences]\n        manual_metrics = _strict_entity_metrics(gold_tag_sequences, pred_tag_sequences, entity_types)\n        strict_metrics, method = _nervaluate_strict_metrics(\n            gold_tag_sequences, pred_tag_sequences, entity_types, manual_metrics)\n        LOGGER.info(f\'Entity Strict method: {method}\')\n        report_text = (\'CoNLL evaluation (legacy flattened-token behavior):\\n\'\n                       + conll_log.getvalue() + \'\\n\'\n                       \'Classification report (BIO token level):\\n\' + report + \'\\n\'\n                       f\'Entity strict method: {method}\\n\')\n        _save_test_artifacts(seed_dir, gold_sequences, pred_sequences, eval_golds, eval_preds,\n                             (prec, rec, f1), seed, model_choice, model_name,\n                             report_text, strict_metrics, method,\n                             token_sequences if source_examples is not None else None)\n        return prec, rec, f1\n    return prec, rec, f1\n',
    'src/models/__init__.py': "from .base import SupConOutput, LinearProbeOutput\nfrom .projection import ProjectionHead\nfrom .losses import TokenSupConLoss\nfrom .linear_probe import TokenEncoderSupCon, TokenEncoderLinearProbe\n\n# Keep legacy import names available for the original command-line entry point.\nPhoBertSupCon = TokenEncoderSupCon\nPhoBertLinearProbe = TokenEncoderLinearProbe\n\n__all__ = [\n    'SupConOutput', 'LinearProbeOutput', 'ProjectionHead', 'TokenSupConLoss',\n    'TokenEncoderSupCon', 'TokenEncoderLinearProbe', 'PhoBertSupCon', 'PhoBertLinearProbe'\n]\n",
    'src/models/base.py': 'from typing import Optional, List, Tuple, Any\nfrom collections import OrderedDict\nimport torch\n\n\nclass SupConOutput(OrderedDict):\n    loss: Optional[torch.FloatTensor] = torch.FloatTensor([0.0])\n    num_valid_anchors: int = 0\n\n    def __init__(self, *args, **kwargs):\n        super(SupConOutput, self).__init__(*args, **kwargs)\n        for k, v in kwargs.items():\n            setattr(self, k, v)\n\n    def __getitem__(self, k):\n        if isinstance(k, str):\n            inner_dict = {k: v for (k, v) in self.items()}\n            return inner_dict[k]\n        else:\n            return self.to_tuple()[k]\n\n    def __setattr__(self, name, value):\n        if name in self.keys() and value is not None:\n            super().__setitem__(name, value)\n        super().__setattr__(name, value)\n\n    def __setitem__(self, key, value):\n        super().__setitem__(key, value)\n        super().__setattr__(key, value)\n\n    def to_tuple(self) -> Tuple[Any]:\n        return tuple(self[k] for k in self.keys())\n\n\nclass LinearProbeOutput(OrderedDict):\n    loss: Optional[torch.FloatTensor] = torch.FloatTensor([0.0])\n    logits: Optional[torch.FloatTensor] = None\n    tags: Optional[List[int]] = []\n\n    def __init__(self, *args, **kwargs):\n        super(LinearProbeOutput, self).__init__(*args, **kwargs)\n        for k, v in kwargs.items():\n            setattr(self, k, v)\n\n    def __getitem__(self, k):\n        if isinstance(k, str):\n            inner_dict = {k: v for (k, v) in self.items()}\n            return inner_dict[k]\n        else:\n            return self.to_tuple()[k]\n\n    def __setattr__(self, name, value):\n        if name in self.keys() and value is not None:\n            super().__setitem__(name, value)\n        super().__setattr__(name, value)\n\n    def __setitem__(self, key, value):\n        super().__setitem__(key, value)\n        super().__setattr__(key, value)\n\n    def to_tuple(self) -> Tuple[Any]:\n        return tuple(self[k] for k in self.keys())\n\n',
    'src/models/linear_probe.py': 'import inspect\n\nimport torch\nimport torch.nn as nn\nfrom transformers import AutoModel, AutoConfig, logging\n\nfrom .base import SupConOutput, LinearProbeOutput\nfrom .losses import TokenSupConLoss\nfrom .projection import ProjectionHead\n\nlogging.set_verbosity_error()\n\n\ndef _load_encoder(model_name_or_config):\n    if isinstance(model_name_or_config, str):\n        config = AutoConfig.from_pretrained(model_name_or_config)\n        encoder = AutoModel.from_pretrained(model_name_or_config, config=config)\n    else:\n        config = model_name_or_config\n        encoder = AutoModel.from_config(config)\n    hidden_size = getattr(config, \'hidden_size\', None) or getattr(config, \'dim\', None)\n    if not hidden_size:\n        raise ValueError(f\'Backbone config has no hidden size: {getattr(config, "model_type", type(config).__name__)}\')\n    config.hidden_size = hidden_size\n    return config, encoder\n\n\ndef _encoder_inputs(encoder, input_ids, token_type_ids=None, attention_mask=None):\n    available = inspect.signature(encoder.forward).parameters\n    accepts_kwargs = any(parameter.kind == inspect.Parameter.VAR_KEYWORD for parameter in available.values())\n    inputs = {\'input_ids\': input_ids, \'attention_mask\': attention_mask, \'token_type_ids\': token_type_ids}\n    return {key: value for key, value in inputs.items()\n            if value is not None and (accepts_kwargs or key in available)}\n\n\ndef _last_hidden_state(outputs):\n    return outputs.last_hidden_state if hasattr(outputs, \'last_hidden_state\') else outputs[0]\n\n\nclass TokenEncoderSupCon(nn.Module):\n    """Backbone plus the original flat supervised contrastive projection head."""\n    def __init__(self, model_name_or_config, projection_dim: int = 128,\n                 temperature: float = 0.07, o_label_id: int = 0):\n        super().__init__()\n        self.config, self.encoder = _load_encoder(model_name_or_config)\n        self.projection_head = ProjectionHead(input_dim=self.config.hidden_size,\n                                              hidden_dim=256, output_dim=projection_dim)\n        self.supcon_loss = TokenSupConLoss(temperature=temperature)\n        self.o_label_id = o_label_id\n\n    def encode_word_representations(self, input_ids, valid_ids, token_type_ids=None, attention_mask=None, **kwargs):\n        outputs = self.encoder(**_encoder_inputs(self.encoder, input_ids, token_type_ids, attention_mask))\n        sequence_outputs = _last_hidden_state(outputs)\n        rows = torch.arange(sequence_outputs.size(0), device=sequence_outputs.device).unsqueeze(1)\n        return sequence_outputs[rows, valid_ids]\n\n    def forward(self, input_ids, token_type_ids=None, attention_mask=None,\n                labels=None, valid_ids=None, label_masks=None):\n        word_representations = self.encode_word_representations(\n            input_ids, valid_ids, token_type_ids, attention_mask)\n        if labels is None or label_masks is None:\n            return SupConOutput(loss=word_representations.new_tensor(0.0), num_valid_anchors=0)\n        valid_entity_mask = (label_masks != 0) & (labels != self.o_label_id)\n        num_valid = int(valid_entity_mask.sum().item())\n        if num_valid < 2:\n            return SupConOutput(loss=word_representations.new_tensor(0.0), num_valid_anchors=num_valid)\n        entity_features = self.projection_head(word_representations)[valid_entity_mask]\n        loss = self.supcon_loss(entity_features, labels[valid_entity_mask])\n        return SupConOutput(loss=loss, num_valid_anchors=num_valid)\n\n\nclass TokenEncoderLinearProbe(nn.Module):\n    """Frozen or optional trainable encoder with a single argmax linear head."""\n    def __init__(self, model_name_or_config, num_labels: int = 11, freeze_backbone: bool = True):\n        super().__init__()\n        self.config, self.encoder = _load_encoder(model_name_or_config)\n        self.num_labels = num_labels\n        self.freeze_backbone = freeze_backbone\n        dropout_probability = getattr(self.config, \'hidden_dropout_prob\', getattr(self.config, \'dropout\', 0.1))\n        self.dropout = nn.Dropout(dropout_probability)\n        self.classifier = nn.Linear(self.config.hidden_size, num_labels)\n        self.loss_fct = nn.CrossEntropyLoss(ignore_index=-100)\n        if self.freeze_backbone:\n            self._freeze_encoder()\n\n    def _freeze_encoder(self):\n        for parameter in self.encoder.parameters():\n            parameter.requires_grad = False\n\n    def encode_word_representations(self, input_ids, valid_ids, token_type_ids=None, attention_mask=None, **kwargs):\n        outputs = self.encoder(**_encoder_inputs(self.encoder, input_ids, token_type_ids, attention_mask))\n        sequence_outputs = _last_hidden_state(outputs)\n        rows = torch.arange(sequence_outputs.size(0), device=sequence_outputs.device).unsqueeze(1)\n        return sequence_outputs[rows, valid_ids]\n\n    def forward(self, input_ids, token_type_ids=None, attention_mask=None,\n                labels=None, valid_ids=None, label_masks=None):\n        if self.freeze_backbone:\n            with torch.no_grad():\n                word_representations = self.encode_word_representations(\n                    input_ids, valid_ids, token_type_ids, attention_mask)\n        else:\n            word_representations = self.encode_word_representations(\n                input_ids, valid_ids, token_type_ids, attention_mask)\n        logits = self.classifier(self.dropout(word_representations))\n        active_mask = label_masks != 0\n        predictions = logits.argmax(dim=-1)\n        tags = [predictions[row, :int(active_mask[row].sum().item())].cpu().tolist()\n                for row in range(input_ids.size(0))]\n        loss = None\n        if labels is not None:\n            active_labels = torch.where(active_mask, labels, torch.full_like(labels, -100))\n            loss = self.loss_fct(logits.reshape(-1, self.num_labels), active_labels.reshape(-1))\n        return LinearProbeOutput(loss=loss, logits=logits, tags=tags)\n',
    'src/models/losses.py': 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass TokenSupConLoss(nn.Module):\n    """\n    Supervised Contrastive Loss phẳng chuẩn Google 100% (Khosla et al., NeurIPS 2020).\n    Hoàn toàn KHÔNG có hệ số alpha.\n    \n    Quy tắc phân lớp nhị phân:\n      - Cặp Dương (Positive pairs): Cùng chính xác nhãn BIO (y_i == y_j) -> Trọng số 1.0.\n      - Cặp Âm (Negative pairs): Bất kỳ token nào khác nhãn (y_i != y_j) -> Đẩy dạt ở mẫu số.\n      - Loại trừ chính nó (self-contrast) bằng self_mask.\n    """\n    def __init__(self, temperature: float = 0.07):\n        super(TokenSupConLoss, self).__init__()\n        self.temperature = temperature\n\n    def forward(self, features: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:\n        """\n        Args:\n            features: [N_entity, proj_dim] (ví dụ: [N, 128])\n            labels:   [N_entity] (1D tensor chứa nhãn BIO của token)\n        """\n        if features.size(0) < 2:\n            return features.new_tensor(0.0)\n\n        # 1. Ép kiểu sang float32 để ổn định số học trong môi trường AMP FP16\n        features = features.float()\n        features = F.normalize(features, p=2, dim=1)\n\n        # 2. Ma trận tương đồng Cosine chuẩn hóa theo nhiệt độ tau\n        # Giu float32 ke ca khi ben ngoai bat autocast fp16: matmul trong autocast tra ve fp16 va\n        # masked_fill(-1e9) tran fp16 (PyTorch moi bao loi: value cannot be converted to type at::Half).\n        with torch.autocast(device_type=features.device.type, enabled=False):\n            similarity = torch.matmul(features, features.T) / self.temperature\n        N = features.size(0)\n\n        # 3. Mặt nạ loại trừ chính nó (self-mask)\n        self_mask = torch.eye(N, device=features.device, dtype=torch.bool)\n\n        # 4. Mặt nạ mẫu dương chuẩn Google: 1.0 nếu cùng nhãn, 0.0 nếu khác nhãn (KHÔNG CÓ ALPHA)\n        labels_col = labels.contiguous().view(-1, 1)\n        pos_mask = (torch.eq(labels_col, labels_col.T) & (~self_mask)).float()\n\n        # 5. Khử đường chéo mẫu số bằng -1e9 (tránh 0.0 * (-inf) = NaN)\n        similarity = similarity.masked_fill(self_mask, -1e9)\n\n        # 6. Ổn định số thực: trừ max trên mỗi hàng\n        logits_max, _ = torch.max(similarity, dim=1, keepdim=True)\n        logits = similarity - logits_max.detach()\n\n        # 7. Tính xác suất log trên toàn bộ mẫu số\n        exp_logits = torch.exp(logits) * (~self_mask).float()\n        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True).clamp_min(1e-12))\n\n        # 8. Gán đường chéo bằng 0.0\n        log_prob = log_prob.masked_fill(self_mask, 0.0)\n\n        # 9. Lọc các anchor hợp lệ có ít nhất 1 mẫu dương trong batch\n        pos_count = pos_mask.sum(dim=1)\n        valid_anchor = pos_count > 0\n\n        if not valid_anchor.any():\n            return features.new_tensor(0.0)\n\n        # 10. Kỳ vọng log-likelihood trung bình trên các cặp dương (chuẩn SupContrast)\n        mean_log_prob_pos = (pos_mask * log_prob).sum(dim=1) / pos_count.clamp_min(1e-12)\n\n        # 11. Hàm mất mát SupCon scalar\n        loss = -mean_log_prob_pos[valid_anchor].mean()\n        if torch.isnan(loss) or torch.isinf(loss):\n            return features.new_tensor(0.0)\n        return loss\n',
    'src/models/projection.py': 'import torch\nimport torch.nn as nn\n\n\nclass ProjectionHead(nn.Module):\n    """\n    Projection Head for Supervised Contrastive Learning (Khosla et al., NeurIPS 2020)\n    Projects backbone contextual representations to a normalized contrastive space.\n    MLP Architecture: Linear(input_dim -> hidden_dim) -> ReLU -> Linear(hidden_dim -> output_dim).\n    """\n    def __init__(self, input_dim: int, hidden_dim: int = 256, output_dim: int = 128):\n        super(ProjectionHead, self).__init__()\n        self.mlp = nn.Sequential(\n            nn.Linear(input_dim, hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, output_dim)\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.mlp(x)\n\n',
    'run_seed.py': 'import argparse\nimport gc\nimport json\nimport os\nimport shutil\nimport sys\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nPROJECT_DIR = Path(__file__).resolve().parent\nsys.path.insert(0, str(PROJECT_DIR))\n\n\ndef _read_json(path):\n    return json.loads(path.read_text(encoding=\'utf-8\'))\n\n\ndef _write_json(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding=\'utf-8\')\n\n\ndef _apply_strict_determinism(config):\n    if not config.get(\'strict_determinism\', False):\n        return\n    import torch\n    torch.use_deterministic_algorithms(True, warn_only=True)\n    for stage in (\'stage1\', \'stage2\'):\n        config[\'training\'][stage][\'fp16\'] = False\n\n\ndef _runtime_metadata():\n    from importlib.metadata import version\n    import torch\n\n    cuda_available = torch.cuda.is_available()\n    cudnn = getattr(getattr(torch.backends, \'cudnn\', None), \'deterministic\', None)\n    return {\n        \'torch_version\': torch.__version__,\n        \'transformers_version\': version(\'transformers\'),\n        \'gpu_name\': torch.cuda.get_device_name(0) if cuda_available else \'CPU (CUDA unavailable)\',\n        \'CUBLAS_WORKSPACE_CONFIG\': os.environ.get(\'CUBLAS_WORKSPACE_CONFIG\'),\n        \'cudnn_deterministic\': cudnn,\n    }\n\n\ndef _stage_arguments(config, stage, micro_batch_size):\n    model = config[\'model\']\n    train = config[\'training\'][stage]\n    seed_dir = Path(config[\'seed_dir\'])\n    values = dict(train)\n    values.update({\n        \'seed\': config[\'seed\'],\n        \'model_choice\': config[\'model_choice\'],\n        \'model_name_or_path\': model[\'model_name_or_path\'],\n        \'use_fast_tokenizer\': model[\'use_fast_tokenizer\'],\n        \'data_dir\': config[\'dataset_dir\'],\n        \'output_dir\': str(seed_dir / (\'stage1\' if stage == \'stage1\' else \'stage2\')),\n        \'micro_batch_size\': micro_batch_size,\n        \'eval_batch_size\': micro_batch_size,\n        \'overwrite_data\': False,\n        \'no_cuda\': False,\n    })\n    if stage == \'stage2\':\n        values[\'pretrained_encoder_path\'] = str(seed_dir / \'stage1\' / \'best_encoder.pt\')\n    return SimpleNamespace(**values)\n\n\ndef _run_pipeline(config, micro_batch_size):\n    from src.helper import set_random_seed\n    from src.trainer import train_stage1_supcon, train_stage2_linear\n\n    seed_dir = Path(config[\'seed_dir\'])\n    _apply_strict_determinism(config)\n    set_random_seed(config[\'seed\'])\n    config[\'runtime\'] = _runtime_metadata()\n    _write_json(seed_dir / \'config.json\', config)\n    stage1_args = _stage_arguments(config, \'stage1\', micro_batch_size)\n    encoder_path = train_stage1_supcon(stage1_args)\n    stage2_args = _stage_arguments(config, \'stage2\', micro_batch_size)\n    stage2_args.pretrained_encoder_path = encoder_path\n    metrics = train_stage2_linear(stage2_args)\n    stage1_history = _read_json(seed_dir / \'stage1\' / \'history.json\')\n    stage2_history = _read_json(seed_dir / \'stage2\' / \'history.json\')\n    history = {\n        \'seed\': config[\'seed\'],\n        \'stage1\': stage1_history[\'stage1\'],\n        \'stage2\': stage2_history[\'stage2\'],\n        \'stage1_best_epoch\': stage1_history[\'best_epoch\'],\n        \'stage2_best_epoch\': stage2_history[\'best_epoch\'],\n        \'stage1_epochs_run\': stage1_history[\'epochs_run\'],\n        \'stage2_epochs_run\': stage2_history[\'epochs_run\'],\n        \'stage1_duration_seconds\': stage1_history[\'total_duration_seconds\'],\n        \'stage2_duration_seconds\': stage2_history[\'total_duration_seconds\'],\n        \'total_duration_seconds\': stage1_history[\'total_duration_seconds\'] + stage2_history[\'total_duration_seconds\'],\n    }\n    _write_json(seed_dir / \'history.json\', history)\n    metrics_path = seed_dir / \'test_metrics.json\'\n    test_metrics = _read_json(metrics_path)\n    test_metrics.update({\n        \'effective_batch_size\': config[\'training\'][\'stage1\'][\'batch_size\'],\n        \'micro_batch_size\': micro_batch_size,\n        \'gradient_accumulation_steps\': config[\'training\'][\'stage1\'][\'batch_size\'] // micro_batch_size,\n        \'oom_retries\': config.get(\'oom_retries\', []),\n    })\n    _write_json(metrics_path, test_metrics)\n    return metrics\n\n\ndef _is_cuda_oom(error):\n    try:\n        import torch\n        if isinstance(error, torch.cuda.OutOfMemoryError):\n            return True\n    except (ImportError, AttributeError):\n        pass\n    return \'cuda out of memory\' in str(error).lower()\n\n\ndef _clear_attempt_outputs(seed_dir):\n    for name in (\'stage1\', \'stage2\'):\n        shutil.rmtree(seed_dir / name, ignore_errors=True)\n    artifact_names = (\n        \'history.json\', \'test_metrics.json\', \'test_predictions.json\', \'test_report.txt\',\n        \'advanced_eval_table.csv\', \'advanced_eval_table.json\',\n        \'paper_table1_bio.csv\', \'paper_table2_entity.csv\', \'failure.json\'\n    )\n    for name in artifact_names:\n        path = seed_dir / name\n        if path.exists():\n            path.unlink()\n\n\ndef run(config_path):\n    config_path = Path(config_path).resolve()\n    config = _read_json(config_path)\n    seed_dir = Path(config[\'seed_dir\']).resolve()\n    seed_dir.mkdir(parents=True, exist_ok=True)\n    config[\'oom_retries\'] = []\n    micro_batch_size = int(config[\'max_micro_batch_size\'])\n    effective_batch_size = int(config[\'training\'][\'stage1\'][\'batch_size\'])\n    if effective_batch_size % micro_batch_size:\n        raise ValueError(\'Initial micro-batch size must divide effective batch size.\')\n\n    while True:\n        config[\'micro_batch_size\'] = micro_batch_size\n        config[\'gradient_accumulation_steps\'] = effective_batch_size // micro_batch_size\n        _write_json(seed_dir / \'config.json\', config)\n        try:\n            metrics = _run_pipeline(config, micro_batch_size)\n            print(f"Completed seed={config[\'seed\']} with micro={micro_batch_size}, accum={effective_batch_size // micro_batch_size}")\n            return metrics\n        except Exception as error:\n            if not _is_cuda_oom(error) or micro_batch_size <= 1:\n                _write_json(seed_dir / \'failure.json\', {\n                    \'seed\': config[\'seed\'], \'error_type\': type(error).__name__,\n                    \'error\': str(error), \'oom_retries\': config[\'oom_retries\'],\n                })\n                raise\n            previous_micro = micro_batch_size\n            micro_batch_size //= 2\n            config[\'oom_retries\'].append({\'from_micro_batch\': previous_micro,\n                                          \'to_micro_batch\': micro_batch_size,\n                                          \'effective_batch_size\': effective_batch_size})\n            print(f\'CUDA OOM at micro-batch {previous_micro}; restarting both stages from scratch with micro-batch {micro_batch_size}.\')\n            _clear_attempt_outputs(seed_dir)\n            import torch\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n            gc.collect()\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--config\', required=True)\n    args = parser.parse_args()\n    run(args.config)\n\n\nif __name__ == \'__main__\':\n    main()\n',
}
for relative_path, source_text in SOURCE_FILES.items():
    target = (PROJECT_DIR / relative_path).resolve()
    if PROJECT_DIR not in target.parents:
        raise ValueError(f"Đường dẫn source nằm ngoài PROJECT_DIR: {target}")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source_text, encoding="utf-8")
print(f"Đã ghi {len(SOURCE_FILES)} file source vào {PROJECT_DIR}")
for relative_path in SOURCE_FILES:
    print("-", (PROJECT_DIR / relative_path).resolve())

## 3. Tải dữ liệu đã pin và kiểm tra số câu
Chọn đúng ba file CoNLL theo cách của notebook XLM-R softmax, sao chép thành tên mà `dataset.py` đọc.

In [ ]:
fetch_script = r"""
import json, sys
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
repo_id, revision, output_root = sys.argv[1:]
info = HfApi().dataset_info(repo_id, revision=revision)
sha = info.sha
if sha != revision:
    raise RuntimeError(f"Dataset SHA mismatch: requested={revision}, received={sha}")
files = [item.rfilename for item in info.siblings]
choices = {
    "train": "train_data_dedup_70_15_15.txt",
    "dev": "dev_data_dedup_70_15_15.txt",
    "test": "test_data_dedup_70_15_15.txt",
}
selected = {}
for split, basename in choices.items():
    matches = [name for name in files if Path(name).name == basename]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {split} file named {basename}; found {matches}")
    selected[split] = matches[0]
root = Path(snapshot_download(repo_id=repo_id, repo_type="dataset", revision=sha,
                              local_dir=str(Path(output_root) / sha),
                              allow_patterns=list(selected.values())))
paths = {split: str((root / name).resolve()) for split, name in selected.items()}
for split, path in paths.items():
    if not Path(path).is_file() or Path(path).stat().st_size == 0:
        raise RuntimeError(f"Missing/empty {split}: {path}")
print(json.dumps({"sha": sha, "files": paths}, ensure_ascii=False))
"""
fetch = subprocess.run(
    [sys.executable, "-c", fetch_script, HF_DATASET_REPO, HF_DATASET_REVISION, str(HF_DATASET_ROOT)],
    env=ENV, capture_output=True, text=True, check=True,
)
DATA_MANIFEST = json.loads(fetch.stdout.strip().splitlines()[-1])
assert DATA_MANIFEST["sha"] == HF_DATASET_REVISION
DATASET_DIR.mkdir(parents=True, exist_ok=True)

def _sha256(path):
    import hashlib
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_paths = DATA_MANIFEST["files"]
targets = {split: DATASET_DIR / f"{split}_data.txt" for split in ("train", "dev", "test")}
data_changed = any(
    not targets[split].is_file() or _sha256(targets[split]) != _sha256(source_paths[split])
    for split in targets
)
if data_changed:
    for cached_file in DATASET_DIR.glob("*.cached"):
        cached_file.unlink()
    print("Dữ liệu thay đổi; đã xóa cache .cached cũ.")
for split, source in source_paths.items():
    shutil.copy2(source, targets[split])

DATASET_MANIFEST = {
    "repo": HF_DATASET_REPO,
    "revision": HF_DATASET_REVISION,
    "files": {split: str(path.resolve()) for split, path in targets.items()},
}
(WORK_ROOT / "dataset_manifest.json").write_text(
    json.dumps(DATASET_MANIFEST, ensure_ascii=False, indent=2), encoding="utf-8"
)

def count_conll_sentences(path):
    count, inside = 0, False
    with Path(path).open(encoding="utf-8-sig") as stream:
        for line in stream:
            if line.strip():
                inside = True
            elif inside:
                count += 1
                inside = False
    return count + int(inside)

for split, path in DATA_MANIFEST["files"].items():
    actual = count_conll_sentences(targets[split])
    expected = EXPECTED_SENTENCES[split]
    assert actual == expected, f"{split}: có {actual} câu, cần {expected}"
    print(f"{split}: {actual:,} câu — {targets[split]}")

## 4. Smoke test một batch
Dùng backbone nhẹ `distilphobert`, 20 câu đầu và CPU; kiểm tra tokenizer, SupCon forward và linear-probe CE/backward trước khi chạy 5 seed.

In [ ]:
import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from src.constant import LABEL2ID_PAP_NER, LABEL_MAPPING
from src.dataset import build_dataset
from src.models import TokenEncoderLinearProbe, TokenEncoderSupCon

SMOKE_MODEL_CHOICE = "distilphobert"
SMOKE_MODEL = MODEL_REGISTRY[SMOKE_MODEL_CHOICE]
SMOKE_DIR = (WORK_ROOT / "smoke").resolve()
SMOKE_DATA_DIR = (SMOKE_DIR / "Dataset").resolve()
SMOKE_DATA_DIR.mkdir(parents=True, exist_ok=True)

def first_sentence_blocks(path, limit=20):
    blocks, current = [], []
    for line in Path(path).read_text(encoding="utf-8-sig").splitlines():
        if line.strip():
            current.append(line)
        elif current:
            blocks.append("\n".join(current))
            current = []
            if len(blocks) >= limit:
                break
    if current and len(blocks) < limit:
        blocks.append("\n".join(current))
    return "\n\n".join(blocks) + "\n\n"

(SMOKE_DATA_DIR / "train_data.txt").write_text(
    first_sentence_blocks(DATASET_DIR / "train_data.txt", 20), encoding="utf-8"
)
smoke_tokenizer = AutoTokenizer.from_pretrained(
    SMOKE_MODEL["model_name_or_path"], use_fast=SMOKE_MODEL["use_fast_tokenizer"]
)
smoke_dataset = build_dataset(
    SMOKE_DATA_DIR, smoke_tokenizer, LABEL2ID_PAP_NER,
    LABEL_MAPPING["pap_ner"]["header"], dtype="train", max_seq_len=MAX_SEQ_LENGTH,
    overwrite_data=True,
)
smoke_loader = DataLoader(smoke_dataset, batch_size=min(4, len(smoke_dataset)), shuffle=False)
smoke_batch = next(iter(smoke_loader))
smoke_device = torch.device("cpu")
smoke_batch = {key: value.to(smoke_device) for key, value in smoke_batch.items()}

supcon_model = TokenEncoderSupCon(SMOKE_MODEL["model_name_or_path"], projection_dim=128, temperature=0.07).to(smoke_device)
supcon_model.train()
supcon_output = supcon_model(**smoke_batch)
assert torch.isfinite(supcon_output.loss), f"SupCon loss không hữu hạn: {supcon_output.loss}"
if supcon_output.loss.requires_grad and supcon_output.loss.item() > 0:
    supcon_output.loss.backward()

probe_model = TokenEncoderLinearProbe(
    SMOKE_MODEL["model_name_or_path"], num_labels=len(LABEL2ID_PAP_NER), freeze_backbone=True
).to(smoke_device)
probe_model.train()
probe_model.encoder.eval()
probe_output = probe_model(**smoke_batch)
assert torch.isfinite(probe_output.loss), f"CE loss không hữu hạn: {probe_output.loss}"
probe_output.loss.backward()
print(f"Smoke test qua: {len(smoke_dataset)} câu; model={SMOKE_MODEL['model_name_or_path']}; device=CPU")

## 5. Chạy 5 seed, mỗi seed một subprocess
Mỗi tiến trình chạy lại Stage 1 rồi Stage 2. Nếu CUDA OOM, tiến trình tự xóa kết quả của lần thử và khởi động lại cả hai stage với micro-batch nhỏ hơn; batch hiệu dụng vẫn 64. Python hash, random, NumPy, PyTorch, CUDA và DataLoader đều được seed. Một số sai khác nhỏ vẫn có thể xuất hiện do FP16 hoặc CUDA atomics; báo cáo nhiều seed theo mean ± std.

In [ ]:
FAILED_SEEDS = []
LIVE_KEYS = ["[STAGE", "early stopping", "Test Strict", "Stage 1 seed", "Stage 2 seed",
             "Traceback", "Error", "OOM", "Using manual strict"]

import re
import matplotlib.pyplot as plt
try:
    from IPython.display import clear_output
except ImportError:
    clear_output = None

_S1 = re.compile(r"\[STAGE 1 - Epoch (\d+)/(\d+)\].*?Train Loss: ([\d.eE+-]+).*?Dev Loss: ([\d.eE+-]+)")
_S2 = re.compile(r"\[STAGE 2 - Epoch (\d+)/(\d+)\].*?Train Loss: ([\d.eE+-]+).*?P: ([\d.eE+-]+).*?R: ([\d.eE+-]+).*?F1: ([\d.eE+-]+)")


class LiveTraining:
    """Vẽ lại biểu đồ mỗi khi có epoch mới (Stage 1: SupCon loss; Stage 2: CE loss và dev Strict P/R/F1)."""

    def __init__(self, seed):
        self.seed = seed
        self.s1 = []  # (epoch, train, dev)
        self.s2 = []  # (epoch, train, p, r, f1)
        self.total1 = self.total2 = None
        self.notes = []

    def feed(self, line):
        if not line or "it/s" in line:
            return
        match = _S1.search(line)
        if match:
            self.total1 = int(match.group(2))
            self.s1.append((int(match.group(1)), float(match.group(3)), float(match.group(4))))
            self.notes.append(line[-150:])
            return self.draw()
        match = _S2.search(line)
        if match:
            self.total2 = int(match.group(2))
            self.s2.append((int(match.group(1)),) + tuple(float(match.group(i)) for i in (3, 4, 5, 6)))
            self.notes.append(line[-150:])
            return self.draw()
        if any(key in line for key in LIVE_KEYS):
            self.notes.append(line[-150:])
            print(f"[seed {self.seed}] {line}", flush=True)

    def draw(self):
        if clear_output is not None:
            clear_output(wait=True)
        e1 = self.s1[-1][0] if self.s1 else 0
        e2 = self.s2[-1][0] if self.s2 else 0
        print(f"Seed {self.seed} | Stage 1: epoch {e1}/{self.total1 or '?'} | Stage 2: epoch {e2}/{self.total2 or '?'}")
        print("\n".join(self.notes[-6:]))
        fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
        if self.s1:
            xs = [r[0] for r in self.s1]
            axes[0].plot(xs, [r[1] for r in self.s1], marker="o", label="train")
            axes[0].plot(xs, [r[2] for r in self.s1], marker="o", label="dev")
            axes[0].legend()
        axes[0].set_title("Stage 1: SupCon loss"); axes[0].set_xlabel("epoch")
        if self.s2:
            xs = [r[0] for r in self.s2]
            axes[1].plot(xs, [r[1] for r in self.s2], marker="o", color="tab:red")
            for index, label in ((2, "P"), (3, "R"), (4, "F1")):
                axes[2].plot(xs, [r[index] for r in self.s2], marker="o", label=label)
            axes[2].legend()
        axes[1].set_title("Stage 2: train loss (CE)"); axes[1].set_xlabel("epoch")
        axes[2].set_title("Stage 2: dev Strict P/R/F1"); axes[2].set_xlabel("epoch")
        plt.tight_layout()
        plt.show()
        plt.close(fig)
RUN_SEED_SCRIPT = (PROJECT_DIR / "run_seed.py").resolve()

for seed in SEEDS:
    seed_dir = (RUN_ROOT / f"seed_{seed}").resolve()
    seed_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = seed_dir / "test_metrics.json"
    if metrics_path.is_file():
        print(f"Seed {seed}: đã có test_metrics.json, bỏ qua để resume.")
        continue

    launch_config = {
        "seed": seed,
        "model_choice": MODEL_CHOICE,
        "model": SELECTED_MODEL,
        "dataset_repo": HF_DATASET_REPO,
        "dataset_revision": HF_DATASET_REVISION,
        "dataset_dir": str(DATASET_DIR),
        "seed_dir": str(seed_dir),
        "max_micro_batch_size": MAX_MICRO_BATCH_SIZE,
        "strict_determinism": STRICT_DETERMINISM,
        "training": TRAINING_CONFIG,
    }
    launch_path = (seed_dir / "launch_config.json").resolve()
    launch_path.write_text(json.dumps(launch_config, ensure_ascii=False, indent=2), encoding="utf-8")
    seed_env = dict(ENV)
    seed_env.update({
        "PYTHONHASHSEED": str(seed),
        "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
        "PYTHONUNBUFFERED": "1",
    })
    log_path = seed_dir / "seed_process.log"
    print(f"Khởi chạy seed {seed} — log: {log_path}")
    # Chạy seed và in trực tiếp các dòng tiến độ (loss/F1 từng epoch); log đầy đủ vẫn ghi vào file.
    live = LiveTraining(seed)
    with log_path.open("a", encoding="utf-8") as log_stream:
        process = subprocess.Popen(
            [sys.executable, str(RUN_SEED_SCRIPT), "--config", str(launch_path)],
            cwd=str(seed_dir), env=seed_env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1,
        )
        for raw_line in process.stdout:
            log_stream.write(raw_line)
            log_stream.flush()
            live.feed(raw_line.strip())
        process.wait()
    class _Result:
        returncode = process.returncode
    result = _Result()
    if result.returncode != 0 or not metrics_path.is_file():
        FAILED_SEEDS.append({"seed": seed, "returncode": result.returncode,
                             "log": str(log_path), "failure": str(seed_dir / "failure.json")})
        print(f"Seed {seed} lỗi (returncode={result.returncode}); các seed tiếp theo vẫn chạy.")
        if (seed_dir / "failure.json").exists():
            print((seed_dir / "failure.json").read_text(encoding="utf-8"))
    else:
        print(f"Seed {seed} hoàn tất.")
    tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-12:]
    print("\n".join(tail))

(RUN_ROOT / "failed_seeds.json").write_text(
    json.dumps(FAILED_SEEDS, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Seeds lỗi:", FAILED_SEEDS)

## 6. Xem kết quả từng seed
Bảng epoch, Bảng 1, Bảng 2 và report đầy đủ được đọc từ các file đã lưu, nên cell này dùng được cả khi tiếp tục notebook sau khi resume.

In [ ]:
from IPython.display import display

ARTIFACT_NAMES = [
    "config.json", "history.json", "test_metrics.json", "test_predictions.json",
    "test_report.txt", "advanced_eval_table.csv", "advanced_eval_table.json",
    "paper_table1_bio.csv", "paper_table2_entity.csv", "stage1/best_encoder.pt",
    "stage2/best_linear_probe.pt", "stage1/history.json", "stage2/history.json",
]
for seed in SEEDS:
    seed_dir = RUN_ROOT / f"seed_{seed}"
    metrics_path = seed_dir / "test_metrics.json"
    print(f"===== SEED {seed} =====")
    config_path = seed_dir / "config.json"
    if not config_path.is_file():
        config_path = seed_dir / "launch_config.json"
    config = json.loads(config_path.read_text(encoding="utf-8")) if config_path.is_file() else {}
    runtime = config.get("runtime", {})
    if not runtime:
        runtime = {
            "torch_version": torch.__version__,
            "transformers_version": transformers.__version__,
            "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (CUDA unavailable)",
            "CUBLAS_WORKSPACE_CONFIG": os.environ.get("CUBLAS_WORKSPACE_CONFIG", ":4096:8"),
            "cudnn_deterministic": getattr(torch.backends.cudnn, "deterministic", None),
        }
    print("Runtime:", json.dumps({
        "torch.__version__": runtime.get("torch_version"),
        "transformers.__version__": runtime.get("transformers_version"),
        "GPU": runtime.get("gpu_name"),
        "CUBLAS_WORKSPACE_CONFIG": runtime.get("CUBLAS_WORKSPACE_CONFIG"),
        "cudnn.deterministic": runtime.get("cudnn_deterministic"),
    }, ensure_ascii=False))
    print("Ghi chú: cùng seed vẫn có thể lệch nhỏ do fp16 và CUDA atomics; báo cáo nhiều seed theo mean ± std.")
    if not metrics_path.is_file():
        print("Không có kết quả hoàn chỉnh.")
        continue
    history = json.loads((seed_dir / "history.json").read_text(encoding="utf-8"))
    print(json.dumps({
        "model_choice": config["model_choice"],
        "checkpoint": config["model"]["model_name_or_path"],
        "dataset": config["dataset_repo"],
        "dataset_revision": config["dataset_revision"],
        "seed": config["seed"],
        "effective_batch_size": config["training"]["stage1"]["batch_size"],
        "micro_batch_size": config.get("micro_batch_size", config.get("max_micro_batch_size")),
        "gradient_accumulation_steps": config.get("gradient_accumulation_steps", 1),
        "stage1": config["training"]["stage1"],
        "stage2": config["training"]["stage2"],
    }, ensure_ascii=False, indent=2))
    print(f"Stage 1: {history['stage1_epochs_run']} epoch; best={history['stage1_best_epoch']}; {history['stage1_duration_seconds']:.1f}s")
    print(f"Stage 2: {history['stage2_epochs_run']} epoch; best={history['stage2_best_epoch']}; {history['stage2_duration_seconds']:.1f}s")
    print("Stage 1 loss / epoch")
    display(pd.DataFrame(history["stage1"])[["epoch", "train_loss", "val_loss", "duration_seconds"]])
    print("Stage 2 loss / dev Strict P/R/F1 / epoch")
    display(pd.DataFrame(history["stage2"])[["epoch", "train_loss", "dev_precision", "dev_recall", "dev_f1", "duration_seconds"]])
    import matplotlib.pyplot as plt
    from IPython.display import Image
    stage1_df = pd.DataFrame(history["stage1"])
    stage2_df = pd.DataFrame(history["stage2"])
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(stage1_df["epoch"], stage1_df["train_loss"], marker="o", label="train")
    axes[0].plot(stage1_df["epoch"], stage1_df["val_loss"], marker="o", label="val")
    axes[0].set_title(f"Seed {seed} — Stage 1: SupCon loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[1].plot(stage2_df["epoch"], stage2_df["train_loss"], marker="o", color="tab:red")
    axes[1].set_title(f"Seed {seed} — Stage 2: train loss (CE)"); axes[1].set_xlabel("epoch")
    for column, label in (("dev_precision", "P"), ("dev_recall", "R"), ("dev_f1", "F1")):
        axes[2].plot(stage2_df["epoch"], stage2_df[column], marker="o", label=label)
    axes[2].set_title(f"Seed {seed} — Stage 2: dev Strict P/R/F1"); axes[2].set_xlabel("epoch"); axes[2].legend()
    plt.tight_layout(); plt.show()
    confusion_path = seed_dir / "stage2" / "confusion_matrix.png"
    if confusion_path.is_file():
        print("Ma trận nhầm lẫn (test, chuẩn hóa theo hàng)")
        display(Image(filename=str(confusion_path), width=700))
    print("Bảng 1 — BIO-level")
    display(pd.read_csv(seed_dir / "paper_table1_bio.csv"))
    print("Bảng 2 — Entity-level Strict")
    display(pd.read_csv(seed_dir / "paper_table2_entity.csv"))
    print("Classification report và log conlleval đầy đủ")
    print((seed_dir / "test_report.txt").read_text(encoding="utf-8"))
    print("File đã lưu:")
    for name in ARTIFACT_NAMES:
        path = seed_dir / name
        if path.is_file() and (INCLUDE_CHECKPOINTS or not path.name.startswith("best_")):
            print("-", path.resolve())

## 7. Gộp seed và tính mean ± std
Độ lệch chuẩn là mẫu (`ddof=1`). Các dòng seed thành công và dòng tổng hợp cùng được lưu trong CSV; `summary.json` giữ mean/std dạng số.

In [ ]:
def _collect_seed_table(filename):
    rows = []
    for seed in SEEDS:
        path = RUN_ROOT / f"seed_{seed}" / filename
        if path.is_file():
            row = pd.read_csv(path).iloc[0].to_dict()
            rows.append(row)
    return pd.DataFrame(rows)

def _append_mean_std(table, metric_columns):
    if table.empty:
        return table, {}
    values = {}
    summary = {}
    for column in metric_columns:
        numeric = pd.to_numeric(table[column], errors="coerce").dropna().to_numpy(dtype=float)
        mean = float(np.mean(numeric)) if len(numeric) else None
        std = float(np.std(numeric, ddof=1)) if len(numeric) > 1 else None
        values[column] = f"{mean:.2f} ± {std:.2f}" if std is not None else (f"{mean:.2f} ± NA" if mean is not None else "")
        summary[column] = {"mean": mean, "std_ddof_1": std, "n": int(len(numeric))}
    for column in table.columns:
        if column not in values:
            values[column] = "Mean ± Std" if column == "Ghi chú" else (MODEL_CHOICE if column == "Phiên bản" else (SELECTED_MODEL["model_name_or_path"] if column == "Model" else ""))
    return pd.concat([table, pd.DataFrame([values], columns=table.columns)], ignore_index=True), summary

paper1_seeds = _collect_seed_table("paper_table1_bio.csv")
paper2_seeds = _collect_seed_table("paper_table2_entity.csv")
paper1_metrics = [column for column in paper1_seeds.columns if "(%)" in column]
paper2_metrics = [column for column in paper2_seeds.columns if "(%)" in column]
paper1_all, paper1_stats = _append_mean_std(paper1_seeds, paper1_metrics)
paper2_all, paper2_stats = _append_mean_std(paper2_seeds, paper2_metrics)

paper1_path = RUN_ROOT / "paper_table1_multiseed.csv"
paper2_path = RUN_ROOT / "paper_table2_multiseed.csv"
paper1_all.to_csv(paper1_path, index=False, encoding="utf-8-sig")
paper2_all.to_csv(paper2_path, index=False, encoding="utf-8-sig")
summary = {
    "model_choice": MODEL_CHOICE,
    "model_name_or_path": SELECTED_MODEL["model_name_or_path"],
    "dataset_repo": HF_DATASET_REPO,
    "dataset_revision": HF_DATASET_REVISION,
    "requested_seeds": SEEDS,
    "successful_seeds": [seed for seed in SEEDS if (RUN_ROOT / f"seed_{seed}" / "test_metrics.json").is_file()],
    "failed_seeds": FAILED_SEEDS,
    "table1_mean_std": paper1_stats,
    "table2_mean_std": paper2_stats,
    "std_ddof": 1,
}
summary_path = RUN_ROOT / "summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("Bảng 1 gộp seed + mean ± std")
display(paper1_all)
print("Bảng 2 gộp seed + mean ± std")
display(paper2_all)
print("Đã lưu:", paper1_path.resolve(), paper2_path.resolve(), summary_path.resolve())

## 8. Nén RUN_ROOT
Tạo một ZIP để tải một lần; theo mặc định bỏ checkpoint `best_*.pt` để giảm dung lượng.

In [ ]:
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, mode="w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
    for path in RUN_ROOT.rglob("*"):
        if not path.is_file():
            continue
        if not INCLUDE_CHECKPOINTS and path.name.startswith("best_") and path.suffix == ".pt":
            continue
        archive.write(path, arcname=str(path.relative_to(RUN_ROOT)))
size_mib = ZIP_PATH.stat().st_size / (1024 ** 2)
print(f"ZIP: {ZIP_PATH.resolve()}")
print(f"Dung lượng: {size_mib:.2f} MiB")